# Baseball Swing Analysis — R-Angle CNN Pipeline (Colab)
### Full videos → Annotate → Train single R-handed CNN → Evaluate

**Key facts baked into this notebook:**
- All videos are **right-handed batters**, ball comes **from the right** (left-side pitcher, standard broadcast angle)
- Input clips are **full-length ~6-11s** — no pre-trimming required
- Contact detection uses a **30–85% temporal gate** to skip load/follow-through phases
- The CNN takes a **3-frame temporal stack** at contact (t-2, t, t+2 frames stacked as 9 channels) — this encodes bat velocity direction, the strongest single predictor of launch angle
- One model: `cnn_R.pt` — no L-model training

**Pipeline stages:**
1. Mount Drive + configure paths
2. Install dependencies
3. Write processing scripts
4. Download MediaPipe model
5. YOLO: ball/bat detection → `.yolo.json` sidecars
6. MediaPipe: pose + launch angle → `_result.json` + `batch_results.csv`
7. Annotate all videos → `_annotated.mp4`
8. Extract 3-frame CNN dataset
9. Train `cnn_R.pt` (ResNet-18, 9-channel input)
10. Evaluate + plot
11. Inference on new videos

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# ── Configure your Drive paths here ─────────────────────────────────────────
# Point DRIVE_VIDEO_DIR at whatever folder contains your .mp4 files.
# The pipeline scans recursively, so nested subfolders are fine.
DRIVE_VIDEO_DIR = Path('/content/drive/MyDrive/BigBaseballSwing')

# All outputs (sidecars, results, model, annotated clips) go here:
OUTPUT_DIR  = DRIVE_VIDEO_DIR / 'outputs'
MODEL_DIR   = DRIVE_VIDEO_DIR / 'models'
DATASET_DIR = DRIVE_VIDEO_DIR / 'cnn_dataset'
CNN_R_PATH  = MODEL_DIR / 'cnn_R.pt'

for d in [OUTPUT_DIR, MODEL_DIR, DATASET_DIR/'train', DATASET_DIR/'val']:
    d.mkdir(parents=True, exist_ok=True)

# ── Scan for all .mp4 files (skip already-annotated clips) ───────────────────
VIDEO_EXTS = {'.mp4', '.mov', '.avi', '.mkv'}
all_videos = sorted(
    p for p in DRIVE_VIDEO_DIR.rglob('*')
    if p.is_file()
    and p.suffix.lower() in VIDEO_EXTS
    and '_annotated' not in p.stem
    and 'outputs' not in str(p)
)

print(f"Drive video folder : {DRIVE_VIDEO_DIR}")
print(f"Videos found       : {len(all_videos)}")
for v in all_videos[:8]:
    print(f"  {v.relative_to(DRIVE_VIDEO_DIR)}")
if len(all_videos) > 8:
    print(f"  ... and {len(all_videos)-8} more")

Mounted at /content/drive
Drive video folder : /content/drive/MyDrive/BigBaseballSwing
Videos found       : 1197
  6s_trimmed_videos/Aaron Judge.mp4
  6s_trimmed_videos/Aaron Judge_swing.mp4
  6s_trimmed_videos/Aaron Judge_swing_swing.mp4
  6s_trimmed_videos/Aaron Judge_swing_swing_swing.mp4
  6s_trimmed_videos/Adam Duvall.mp4
  6s_trimmed_videos/Adam Duvall_swing.mp4
  6s_trimmed_videos/Adam Duvall_swing_swing.mp4
  6s_trimmed_videos/Adam Duvall_swing_swing_swing.mp4
  ... and 1189 more


## 2. Install dependencies

In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

pip('ultralytics')
pip('mediapipe')
pip('scipy', 'opencv-python-headless', 'matplotlib', 'scikit-learn', 'tqdm')

import torch, cv2, mediapipe, ultralytics
print(f"PyTorch    : {torch.__version__}  GPU={torch.cuda.is_available()}")
print(f"OpenCV     : {cv2.__version__}")
print(f"MediaPipe  : {mediapipe.__version__}")
print(f"Ultralytics: {ultralytics.__version__}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
PyTorch    : 2.10.0+cu128  GPU=True
OpenCV     : 4.13.0
MediaPipe  : 0.10.35
Ultralytics: 8.4.51


## 3. Write processing scripts to Colab disk

In [ ]:
yolo_src = "\"\"\"\nyolo_processing.py\n==================\nStage 1 of the two-stage baseball swing analysis pipeline.\n\nResponsibility\n--------------\n- Load each swing video\n- Run YOLOv8-nano to detect the baseball and bat in every frame\n- Run a Kalman filter to smooth trajectories and fill occlusion gaps\n- Compute the ball launch angle from the post-contact parabola\n- Write one JSON sidecar file per video containing all detection results\n\nOutput format (one JSON file per video)\n----------------------------------------\n{\n  \"video_path\":            \"/abs/path/to/video.mp4\",\n  \"fps\":                   60.0,\n  \"width\":                 1722,\n  \"height\":                1080,\n  \"contact_frame\":         238,          # frame index where ball momentum reverses\n  \"contact_time_s\":        3.967,\n  \"ball_angle\":            14.3,         # launch angle from ball trajectory (degrees)\n  \"ball_contact_px\":       [841, 512],   # [cx, cy] pixel of first post-contact detection\n  \"post_contact\": [                      # one entry per tracked post-contact frame\n    {\"frame\": 238, \"cx\": 841.2, \"cy\": 512.7, \"r\": 9.1},\n    ...\n  ],\n  \"pre_contact\": [                       # one entry per tracked pre-contact frame (pitch)\n    {\"frame\": 220, \"cx\": 310.5, \"cy\": 490.1, \"r\": 8.4},\n    ...\n  ],\n  \"bat_boxes\": {                         # YOLO bat detections keyed by frame index (string)\n    \"235\": [1020.1, 380.5, 1180.3, 560.2],   # [x1, y1, x2, y2]\n    ...\n  }\n}\n\nIf ball_angle is null, the ball could not be tracked with enough confidence.\nIf bat_boxes is empty, the model has no bat class (base COCO weights \u2014 fine-tune to add it).\n\nEnvironment\n-----------\n  conda create -n yolo_env python=3.10\n  pip install ultralytics numpy\n\nRun\n---\n  python yolo_processing.py\n\nThe script processes every .mp4/.mov in VIDEO_ROOT and writes a .json sidecar\nnext to each video. mediapipe_processing.py reads those sidecars.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nimport logging\nimport math\nimport sys\nfrom pathlib import Path\nfrom typing import Optional\n\nimport cv2\nimport numpy as np\nfrom ultralytics import YOLO\n\nlogging.basicConfig(\n    level=logging.INFO,\n    format=\"%(asctime)s [%(levelname)s] %(message)s\",\n)\nlog = logging.getLogger(__name__)\n\n# ---------------------------------------------------------------------------\n# Configuration  \u2190 edit these, then run: python yolo_processing.py\n# ---------------------------------------------------------------------------\n\n# Pitcher handedness: \"L\" (left-handed) or \"R\" (right-handed).\n# This determines the side of the frame where the ball MUST originate and\n# the direction it MUST travel after contact.\n#\n#   PITCHER_SIDE = \"L\"  \u2192  ball originates from the RIGHT side of the frame\n#                           (left-handed pitcher throws from third-base side)\n#   PITCHER_SIDE = \"R\"  \u2192  ball originates from the LEFT  side of the frame\n#                           (right-handed pitcher throws from first-base side)\n#\n# Set to None to disable directional constraints (original behaviour).\nPITCHER_SIDE: Optional[str] = \"R\"   # \u2190 edit to \"L\", \"R\", or None\n\n# Fraction of frame width that defines the pitcher-side origin zone.\n# Pre-contact ball candidates outside this zone are penalised / rejected.\n# E.g. 0.45 means the first 45 % of width is the \"left origin\" zone.\nPITCHER_SIDE_ZONE_FRAC = 0.45\n\n# Folder containing trimmed swing videos\nVIDEO_ROOT = Path(__file__).parent / \"trimmed_videos\"\n\n# YOLOv8 weights \u2014 \"yolov8n.pt\" downloads automatically on first run (~6 MB)\n# Replace with a fine-tuned baseball model once trained:\n#   YOLO_WEIGHTS = Path(__file__).parent / \"models\" / \"baseball_yolo.pt\"\nYOLO_WEIGHTS = \"yolov8n.pt\"\n\n# Detection confidence threshold (lower = more detections, more false positives)\n# Raised from 0.25 \u2192 0.30: COCO \"sports ball\" class fires on helmets/crowd at low conf\nCONF_THRESHOLD = 0.30\n\n# How many frames to search forward/backward from the contact frame\nPOST_SEARCH_FRAMES = 30\nPRE_SEARCH_FRAMES  = 20\n\n# Kalman: accept up to this many consecutive misses before stopping tracking\nMAX_CONSECUTIVE_MISSES = 5\n\n# Minimum number of post-contact detections required to compute a ball angle\nMIN_DETECTIONS_FOR_ANGLE = 5\n\n# Ball size range in pixels (min_radius, max_radius) \u2014 tunes the bg-sub fallback\nBALL_SIZE_HINT = (8, 40)\n\n# \u2500\u2500 New tracking-quality filters \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\n# Minimum pixel displacement between consecutive confirmed detections.\n# The ball at 60fps travels ~15-30px/frame. Near-static \"detections\" are\n# background noise (crowd logos, jersey numbers, infield dirt).\nMIN_BALL_SPEED_PX_PER_FRAME = 4.0\n\n# Minimum R\u00b2 of a parabolic fit to post-contact centroids.\n# Below this threshold the \"trajectory\" is random noise; ball_angle is nulled.\nTRAJECTORY_MIN_R2 = 0.60\n\n# Velocity-direction cone (degrees). Once the Kalman filter has estimated a\n# velocity, candidates outside this cone relative to the predicted direction\n# are rejected. Prevents the tracker from snapping to crowd blobs on the\n# opposite side of the frame.\nVEL_CONE_DEG = 80.0\n\nVIDEO_EXTENSIONS = {\".mp4\", \".mov\", \".avi\", \".mkv\"}\n\n# ---------------------------------------------------------------------------\n# COCO class IDs\n# ---------------------------------------------------------------------------\nCOCO_SPORTS_BALL = 32   # \"sports ball\" in base COCO weights\n# \"baseball bat\" is NOT in standard COCO \u2014 bat detection requires fine-tuning.\n# After fine-tuning, add: FINE_TUNED_BAT_CLASS = 1  (depends on your data.yaml)\n\n\n# ---------------------------------------------------------------------------\n# Pitcher-side directional helpers\n# ---------------------------------------------------------------------------\n\ndef _pitcher_origin_x_range(width: int) -> Optional[tuple[int, int]]:\n    \"\"\"\n    Return the (x_min, x_max) pixel range where the pitch MUST originate,\n    derived from PITCHER_SIDE and the frame width.\n\n    L pitcher \u2192 ball comes from the RIGHT  (x > (1 - ZONE_FRAC) * width)\n    R pitcher \u2192 ball comes from the LEFT   (x < ZONE_FRAC * width)\n    None      \u2192 no constraint.\n    \"\"\"\n    if PITCHER_SIDE is None or width <= 0:\n        return None\n    zone = int(PITCHER_SIDE_ZONE_FRAC * width)\n    if PITCHER_SIDE.upper() == \"R\":\n        return (0, zone)                 # left quarter of frame\n    else:                                # \"L\"\n        return (width - zone, width)     # right quarter of frame\n\n\ndef _post_contact_direction(pitcher_side: Optional[str]) -> Optional[int]:\n    \"\"\"\n    Return the expected sign of post-contact vx:\n      +1  ball travels LEFT \u2192 RIGHT  (R pitcher: ball came from left, batter hits it back left)\n      -1  ball travels RIGHT \u2192 LEFT  (L pitcher: ball came from right, batter hits it back right)\n    None  if pitcher_side is None.\n\n    NOTE: image x increases left\u2192right, so a ball hit toward right field\n    (for a R-pitcher/L-batter broadcast angle) has vx > 0.\n    Conventionally from a center-field camera the post-contact ball moves\n    *away* from the pitcher's release side, so direction is opposite.\n    \"\"\"\n    if pitcher_side is None:\n        return None\n    # R-pitcher \u2192 ball originated LEFT \u2192 batter drives it toward RIGHT (+vx)\n    # L-pitcher \u2192 ball originated RIGHT \u2192 batter drives it toward LEFT  (-vx)\n    return +1 if pitcher_side.upper() == \"R\" else -1\n\n\ndef _side_bias_score(cx: float, width: int, is_pre_contact: bool) -> float:\n    \"\"\"\n    Return a multiplicative score bonus [1.0 \u2026 2.0] that rewards candidates\n    on the correct side of the frame, and a penalty [0.0 \u2026 1.0] that\n    suppresses candidates on the wrong side.\n\n    Pre-contact  \u2192 bonus for being in the pitcher's origin zone.\n    Post-contact \u2192 bonus for being on the opposite side (ball moving away).\n    \"\"\"\n    if PITCHER_SIDE is None or width <= 0:\n        return 1.0\n\n    origin = _pitcher_origin_x_range(width)\n    if origin is None:\n        return 1.0\n\n    if is_pre_contact:\n        in_zone = origin[0] <= cx <= origin[1]\n    else:\n        # Post-contact: expected zone is the opposite side\n        if PITCHER_SIDE.upper() == \"R\":\n            # ball came from left, now goes right\n            in_zone = cx > width * (1.0 - PITCHER_SIDE_ZONE_FRAC)\n        else:\n            # ball came from right, now goes left\n            in_zone = cx < width * PITCHER_SIDE_ZONE_FRAC\n\n    return 1.8 if in_zone else 0.5   # strongly reward / mildly suppress\n\n\ndef _enforce_vx_direction(\n    velocity: Optional[tuple[float, float]],\n    is_pre_contact: bool,\n) -> bool:\n    \"\"\"\n    Return False if the current Kalman velocity is firmly in the WRONG\n    horizontal direction for the phase of the play (pre/post contact).\n\n    We only reject when the speed is meaningful (\u2265 5 px/frame) and the\n    direction is clearly backwards (cos < \u22120.5 against expected direction).\n    This prevents the tracker from following a background blob that\n    happens to move toward the pitcher while we are tracking the pitch.\n    \"\"\"\n    if PITCHER_SIDE is None or velocity is None:\n        return True   # no constraint\n\n    vx, vy = velocity\n    speed  = math.sqrt(vx ** 2 + vy ** 2)\n    if speed < 5.0:\n        return True   # not fast enough to be certain\n\n    expected_sign = _post_contact_direction(PITCHER_SIDE)\n    if expected_sign is None:\n        return True\n\n    if is_pre_contact:\n        # Pre-contact: ball should be moving toward the batter,\n        # i.e., opposite of the post-contact direction.\n        expected_vx_sign = -expected_sign\n    else:\n        expected_vx_sign = expected_sign\n\n    # Reject only if strongly moving the WRONG way\n    return (vx * expected_vx_sign) >= -speed * 0.5\n\n\n# ---------------------------------------------------------------------------\n# Kalman filter (numpy only \u2014 no extra deps)\n# ---------------------------------------------------------------------------\n\nclass BallKalmanFilter:\n    \"\"\"\n    2-D constant-acceleration Kalman filter for a baseball in flight.\n\n    State vector : [cx, cy, vx, vy, ax, ay]\n    Observation  : [cx, cy]\n\n    Why Kalman instead of a smoother?\n    - The ball disappears for 2-5 frames immediately after contact (occluded\n      by the bat/body). That is exactly the window that determines launch angle.\n    - Kalman extrapolates through gaps using physics, then self-corrects when\n      YOLO detects the ball again. Savitzky-Golay cannot do this.\n    \"\"\"\n\n    def __init__(self, fps: float) -> None:\n        dt = 1.0 / fps\n        # State transition matrix (constant-acceleration kinematics)\n        self.F = np.array([\n            [1, 0, dt,  0, 0.5 * dt**2, 0          ],\n            [0, 1,  0, dt, 0,           0.5 * dt**2 ],\n            [0, 0,  1,  0, dt,          0           ],\n            [0, 0,  0,  1,  0,          dt          ],\n            [0, 0,  0,  0,  1,          0           ],\n            [0, 0,  0,  0,  0,          1           ],\n        ], dtype=float)\n        # Observation matrix \u2014 we only measure position\n        self.H = np.zeros((2, 6), dtype=float)\n        self.H[0, 0] = 1.0\n        self.H[1, 1] = 1.0\n        # Process noise (tuned for a fast-moving baseball)\n        self.Q = np.eye(6, dtype=float) * 50.0\n        # Measurement noise (~3px standard deviation \u2192 variance = 9)\n        self.R = np.eye(2, dtype=float) * 9.0\n        # State + covariance (uninitialised until first detection)\n        self.x: Optional[np.ndarray] = None\n        self.P: Optional[np.ndarray] = None\n\n    def initialise(self, cx: float, cy: float) -> None:\n        self.x = np.array([cx, cy, 0.0, 0.0, 0.0, 0.0], dtype=float)\n        self.P = np.eye(6, dtype=float) * 500.0\n\n    def predict(self) -> Optional[tuple[float, float]]:\n        \"\"\"Advance one timestep. Returns predicted (cx, cy), or None if not yet initialised.\"\"\"\n        if self.x is None:\n            return None\n        self.x = self.F @ self.x\n        self.P = self.F @ self.P @ self.F.T + self.Q\n        return float(self.x[0]), float(self.x[1])\n\n    def update(self, cx: float, cy: float) -> tuple[float, float]:\n        \"\"\"Incorporate a new measurement. Returns Kalman-corrected (cx, cy).\"\"\"\n        if self.x is None:\n            self.initialise(cx, cy)\n            return cx, cy\n        z = np.array([cx, cy], dtype=float)\n        y = z - self.H @ self.x\n        S = self.H @ self.P @ self.H.T + self.R\n        K = self.P @ self.H.T @ np.linalg.inv(S)\n        self.x = self.x + K @ y\n        self.P = (np.eye(6) - K @ self.H) @ self.P\n        return float(self.x[0]), float(self.x[1])\n\n    def velocity(self) -> Optional[tuple[float, float]]:\n        \"\"\"Return the current estimated (vx, vy) from the state vector, or None.\"\"\"\n        if self.x is None:\n            return None\n        return float(self.x[2]), float(self.x[3])\n\n\n# ---------------------------------------------------------------------------\n# YOLO detector wrapper\n# ---------------------------------------------------------------------------\n\nclass YOLODetector:\n    \"\"\"\n    Wraps a YOLOv8 model with baseball-specific class resolution.\n\n    With the base COCO weights (yolov8n.pt):\n      - Ball is detected as class 32 (\"sports ball\").\n      - Bat is NOT detected \u2014 COCO has no bat class.\n\n    After fine-tuning on a labelled baseball dataset (see README):\n      - Ball class index comes from your data.yaml (e.g. class 0 = \"ball\").\n      - Bat class index comes from your data.yaml (e.g. class 1 = \"bat\").\n\n    Fine-tuning instructions\n    ------------------------\n    1. Download a labelled dataset from Roboflow Universe:\n          https://universe.roboflow.com/search?q=baseball+bat+ball\n       Export in \"YOLOv8\" format.\n\n    2. Create data.yaml:\n          train: /path/to/dataset/train/images\n          val:   /path/to/dataset/val/images\n          nc: 2\n          names: [ball, bat]\n\n    3. Fine-tune (takes ~30 min on a GPU, ~3 hrs on CPU for 50 epochs):\n          from ultralytics import YOLO\n          model = YOLO(\"yolov8n.pt\")\n          model.train(data=\"data.yaml\", epochs=50, imgsz=640, batch=8)\n\n    4. Point YOLO_WEIGHTS to the output:\n          YOLO_WEIGHTS = \"runs/detect/train/weights/best.pt\"\n    \"\"\"\n\n    def __init__(self, weights: str | Path, conf: float = CONF_THRESHOLD) -> None:\n        self.model = YOLO(str(weights))   # downloads weights if not found locally\n        self.conf  = conf\n        names = self.model.names          # {class_id: class_name}\n\n        # Resolve ball class: prefer fine-tuned \"ball\" label, fall back to COCO 32\n        self.ball_class = next(\n            (k for k, v in names.items() if v.lower() in (\"ball\", \"baseball\", \"sports ball\")),\n            COCO_SPORTS_BALL,\n        )\n        # Resolve bat class: only present in fine-tuned models\n        self.bat_class = next(\n            (k for k, v in names.items() if \"bat\" in v.lower()), None\n        )\n\n        log.info(\n            \"YOLO loaded '%s' | ball_class=%s (%s) | bat_class=%s\",\n            Path(str(weights)).name,\n            self.ball_class,\n            names.get(self.ball_class, \"?\"),\n            f\"{self.bat_class} ({names.get(self.bat_class, '?')})\" if self.bat_class else \"none\",\n        )\n\n    def detect(\n        self,\n        frame: np.ndarray,\n    ) -> tuple[\n        Optional[tuple[float, float, float]],       # (cx, cy, radius) or None\n        Optional[tuple[float, float, float, float]], # (x1, y1, x2, y2) or None\n    ]:\n        \"\"\"\n        Run inference on one BGR frame.\n\n        Returns the highest-confidence ball detection and (if a fine-tuned\n        bat class exists) the highest-confidence bat detection.\n        \"\"\"\n        results = self.model(frame, conf=self.conf, verbose=False)[0]\n\n        best_ball: Optional[tuple[float, float, float]] = None\n        best_bat:  Optional[tuple[float, float, float, float]] = None\n        best_ball_conf = 0.0\n        best_bat_conf  = 0.0\n\n        for box in results.boxes:\n            cls_id = int(box.cls[0])\n            conf   = float(box.conf[0])\n            x1, y1, x2, y2 = (float(v) for v in box.xyxy[0])\n\n            if cls_id == self.ball_class and conf > best_ball_conf:\n                cx = (x1 + x2) / 2\n                cy = (y1 + y2) / 2\n                # Radius: geometric mean of half-widths (handles non-square boxes)\n                r  = math.sqrt(((x2 - x1) * (y2 - y1)) / math.pi)\n                best_ball = (cx, cy, r)\n                best_ball_conf = conf\n\n            elif self.bat_class is not None and cls_id == self.bat_class \\\n                    and conf > best_bat_conf:\n                best_bat = (x1, y1, x2, y2)\n                best_bat_conf = conf\n\n        return best_ball, best_bat\n\n\n# ---------------------------------------------------------------------------\n# Background-subtraction fallback detector\n# ---------------------------------------------------------------------------\n\n# ---------------------------------------------------------------------------\n# Ball detector \u2014 temporal differencing + Hough + Kalman gating\n# ---------------------------------------------------------------------------\n#\n# Why temporal differencing instead of a static background model:\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# The static bg model is built from frames ~30 frames before contact. By the\n# time the ball is in play, the batter's body has moved significantly into\n# those quiet-background positions, creating a massive foreground blob that\n# overwhelms the tiny ball signal.\n#\n# Frame-to-frame differencing compares each frame only to the PREVIOUS frame.\n# Between consecutive frames at 60fps:\n#   - A 95mph fastball moves ~22px  \u2192 appears as a bright streak\n#   - The batter's torso moves ~3-5px \u2192 appears as a faint edge\n#   - The crowd moves ~0-2px         \u2192 appears as nothing\n#\n# This 10:1 motion ratio is the key discriminator. The ball is the fastest\n# small object in the frame by a large margin.\n\n\ndef _score_blob(cnt: np.ndarray, lo: int, hi: int) -> Optional[tuple[float, float, float, float]]:\n    \"\"\"\n    Score a contour as a potential ball candidate.\n    Returns (score, cx, cy, r) or None if the contour fails basic filters.\n\n    Scoring combines:\n      - Circularity  (1.0 = perfect circle; streaks score ~0.3-0.7)\n      - Size match   (how close the radius is to expected ball size)\n      - Area         (larger blobs within size range preferred)\n    \"\"\"\n    area = cv2.contourArea(cnt)\n    # Accept blobs from ball-sized up to motion-streak-sized (3\u00d7 ball area)\n    if not (lo**2 * 0.2 < area < hi**2 * 4.0):\n        return None\n    M = cv2.moments(cnt)\n    if M[\"m00\"] <= 0:\n        return None\n    cx    = M[\"m10\"] / M[\"m00\"]\n    cy    = M[\"m01\"] / M[\"m00\"]\n    r     = math.sqrt(area / math.pi)\n    perim = cv2.arcLength(cnt, True)\n    circ  = 4 * math.pi * area / (perim**2 + 1e-6)\n    mid_r = (lo + hi) / 2.0\n    # Size score: 1.0 when r == mid_r, falls off toward 0 at the limits\n    size_score = max(0.0, 1.0 - abs(r - mid_r) / (hi - lo + 1e-6))\n    score = circ * size_score * math.sqrt(area)\n    return score, cx, cy, r\n\n\ndef _candidates_temporal(\n    gray: np.ndarray,\n    prev_gray: np.ndarray,\n    size_hint: tuple[int, int],\n) -> list[tuple[float, float, float, float]]:\n    \"\"\"\n    Frame-to-frame temporal differencing.\n\n    Compares the current frame to the immediately previous frame.\n    Fast-moving small objects (the ball) produce bright, compact blobs.\n    Slow-moving large objects (the crowd, the batter's body) produce faint,\n    diffuse edges that fall below the threshold or fail the size filter.\n\n    Returns list of (score, cx, cy, r).\n    \"\"\"\n    diff    = cv2.absdiff(gray, prev_gray)\n    # Threshold tuned for 60fps slow-mo: ball moves fast, so diff is bright\n    _, thresh = cv2.threshold(diff, 18, 255, cv2.THRESH_BINARY)\n    kernel  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))\n    thresh  = cv2.morphologyEx(thresh, cv2.MORPH_OPEN,  kernel)\n    thresh  = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)\n\n    cnts, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)\n    lo, hi  = size_hint\n    results: list[tuple[float, float, float, float]] = []\n    for cnt in cnts:\n        scored = _score_blob(cnt, lo, hi)\n        if scored is not None:\n            results.append(scored)\n    return results\n\n\ndef _candidates_hough_on_diff(\n    gray: np.ndarray,\n    prev_gray: np.ndarray,\n    size_hint: tuple[int, int],\n) -> list[tuple[float, float, float, float]]:\n    \"\"\"\n    Hough circle detection on the temporal difference image.\n\n    Using Hough on the diff (not the raw frame) dramatically reduces false\n    positives: background textures, jersey patterns, and crowd don't produce\n    circular blobs in the frame-difference image, but a blurry round ball does.\n\n    Returns list of (score, cx, cy, r).\n    \"\"\"\n    diff   = cv2.absdiff(gray, prev_gray)\n    # Enhance contrast in the diff before Hough\n    diff   = cv2.GaussianBlur(diff, (3, 3), 0)\n    lo, hi = size_hint\n\n    circles = cv2.HoughCircles(\n        diff,\n        cv2.HOUGH_GRADIENT,\n        dp=1.0,\n        minDist=15,\n        param1=40,          # Canny threshold \u2014 lower for diff images\n        param2=12,          # accumulator threshold \u2014 tight to reduce false positives\n        minRadius=max(2, lo // 2),\n        maxRadius=hi + 5,\n    )\n    if circles is None:\n        return []\n\n    mid_r   = (lo + hi) / 2.0\n    results = []\n    for cx, cy, r in circles[0]:\n        size_score = max(0.0, 1.0 - abs(r - mid_r) / (hi - lo + 1e-6))\n        results.append((float(size_score * 15), float(cx), float(cy), float(r)))\n    return results\n\n\ndef _candidates_streak(\n    gray: np.ndarray,\n    prev_gray: np.ndarray,\n    size_hint: tuple[int, int],\n) -> list[tuple[float, float, float, float]]:\n    \"\"\"\n    Ellipse fitting on temporal-diff blobs to catch motion-blurred streaks.\n\n    A fast pitch at 60fps leaves an elongated oval streak 2-4\u00d7 longer than\n    the ball diameter. Fitting an ellipse to diff blobs and accepting those\n    with a plausible aspect ratio (1\u20135\u00d7) and minor axis matching ball size\n    catches the ball when it's too blurred to trigger the circular Hough.\n\n    Returns list of (score, cx, cy, estimated_r).\n    \"\"\"\n    diff    = cv2.absdiff(gray, prev_gray)\n    _, thresh = cv2.threshold(diff, 12, 255, cv2.THRESH_BINARY)\n    # Wide horizontal kernel to connect streak pixels\n    kernel  = cv2.getStructuringElement(cv2.MORPH_RECT, (9, 3))\n    thresh  = cv2.dilate(thresh, kernel, iterations=1)\n    kernel2 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))\n    thresh  = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel2)\n\n    cnts, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)\n    lo, hi  = size_hint\n    results = []\n\n    for cnt in cnts:\n        if len(cnt) < 5:\n            continue\n        area = cv2.contourArea(cnt)\n        # Streaks span 0.5\u20138\u00d7 ball area\n        if not (lo**2 * 0.3 < area < hi**2 * 8.0):\n            continue\n        try:\n            (cx, cy), (ma, mi), _ = cv2.fitEllipse(cnt)\n        except Exception:\n            continue\n        if mi < 1:\n            continue\n        aspect = ma / (mi + 1e-6)\n        if not (0.8 < aspect < 6.0):\n            continue\n        # Minor axis \u2248 ball diameter; r \u2248 minor axis / 2\n        r = mi / 2.0\n        if not (lo * 0.4 < r < hi * 1.6):\n            continue\n        # Best score at aspect ~1.5 (slightly elongated fast ball)\n        aspect_score = 1.0 / (1.0 + abs(aspect - 1.5))\n        results.append((aspect_score * math.sqrt(area), float(cx), float(cy), max(r, lo * 0.5)))\n\n    return results\n\n\ndef _gate_and_select(\n    candidates: list[tuple[float, float, float, float]],\n    kalman_pred: Optional[tuple[float, float]],\n    gate_radius: float,\n    velocity: Optional[tuple[float, float]] = None,\n    vel_cone_deg: float = VEL_CONE_DEG,\n    frame_width: int = 0,\n    is_pre_contact: bool = True,\n) -> Optional[tuple[float, float, float]]:\n    \"\"\"\n    Apply Kalman gate and return the highest-scoring surviving candidate.\n\n    Gate logic:\n    - If no prior (kalman_pred is None): accept all candidates, pick best score.\n    - If prior exists: reject candidates outside gate_radius pixels of prediction.\n    - Velocity cone filter: when Kalman velocity is non-trivial (\u22653 px/frame),\n      additionally reject candidates whose direction from the predicted position\n      deviates by more than vel_cone_deg from the estimated velocity vector.\n      This kills background false positives that survive the spatial gate but\n      are moving in the wrong direction relative to the ball's trajectory.\n    - If all candidates are outside the gate: return None (miss \u2014 Kalman extrapolates).\n    - Pitcher-side bias: candidates on the expected side get a score multiplier;\n      candidates on the wrong side are suppressed.\n\n    This kills background false positives: crowd pixels, jersey numbers, and\n    infield dirt never follow a ballistic trajectory, so they are always\n    outside the gate once the tracker has a prior.\n    \"\"\"\n    if not candidates:\n        return None\n\n    # \u2500\u2500 Pitcher-side bias rescoring \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    if PITCHER_SIDE is not None and frame_width > 0:\n        candidates = [\n            (score * _side_bias_score(cx, frame_width, is_pre_contact), cx, cy, r)\n            for score, cx, cy, r in candidates\n        ]\n\n    if kalman_pred is None:\n        best = max(candidates, key=lambda c: c[0])\n        return best[1], best[2], best[3]\n\n    px, py = kalman_pred\n    gated  = [\n        c for c in candidates\n        if math.sqrt((c[1] - px)**2 + (c[2] - py)**2) <= gate_radius\n    ]\n    if not gated:\n        return None\n\n    # \u2500\u2500 Velocity direction cone filter \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    # Only apply when the estimated speed is meaningful (\u22653 px/frame).\n    # This prevents the tracker from locking onto a background object that\n    # happens to be near the predicted position but is moving the wrong way.\n    if velocity is not None:\n        vx, vy = velocity\n        speed  = math.sqrt(vx**2 + vy**2)\n        if speed >= 3.0:\n            cos_thresh = math.cos(math.radians(vel_cone_deg))\n            direction_gated = []\n            for c in gated:\n                dx = c[1] - px\n                dy = c[2] - py\n                d  = math.sqrt(dx**2 + dy**2)\n                if d < 1.0:\n                    direction_gated.append(c)\n                    continue\n                cos_angle = (dx * vx + dy * vy) / (d * speed)\n                if cos_angle >= cos_thresh:\n                    direction_gated.append(c)\n            # Fall back to spatial-only gate if the cone eliminates everything\n            if direction_gated:\n                gated = direction_gated\n\n    best = max(gated, key=lambda c: c[0])\n    return best[1], best[2], best[3]\n\n\ndef detect_ball(\n    gray: np.ndarray,\n    prev_gray: np.ndarray,\n    size_hint: tuple[int, int],\n    kalman_pred: Optional[tuple[float, float]] = None,\n    gate_radius: float = 60.0,\n    velocity: Optional[tuple[float, float]] = None,\n    frame_width: int = 0,\n    is_pre_contact: bool = True,\n) -> Optional[tuple[float, float, float]]:\n    \"\"\"\n    Three-cue temporal ball detector with Kalman gating.\n\n    All three cues operate on the FRAME-TO-FRAME DIFFERENCE (gray \u2212 prev_gray),\n    not on a static background model. This is the key architectural change:\n\n    Static bg model problems:\n      - Built from frames 30+ before contact \u2192 batter has moved into it\n      - Batter's body (huge blob) overwhelms ball signal\n      - Any scene change (pan, zoom) corrupts the model permanently\n\n    Temporal diff advantages:\n      - Batter moves slowly frame-to-frame \u2192 contributes only faint edges\n      - Ball moves 15-30px/frame \u2192 produces a bright, compact blob\n      - Immune to scene changes (each frame pair is independent)\n      - No model build-up time required\n\n    Cue 1 \u2014 Temporal bg-sub: finds compact bright blobs in the diff\n    Cue 2 \u2014 Hough on diff:   finds circular edges even through blur\n    Cue 3 \u2014 Streak fitting:  catches the ball when it's motion-blurred to an oval\n\n    All candidates are Kalman-gated to reject the ones that are in the right\n    pixel range but the wrong location. The velocity cone filter additionally\n    rejects candidates moving in the wrong direction once tracking is established.\n    Pitcher-side bias re-scores candidates by expected frame-side.\n\n    Returns (cx, cy, radius) or None.\n    \"\"\"\n    candidates: list[tuple[float, float, float, float]] = []\n    candidates.extend(_candidates_temporal(gray, prev_gray, size_hint))\n    candidates.extend(_candidates_hough_on_diff(gray, prev_gray, size_hint))\n    candidates.extend(_candidates_streak(gray, prev_gray, size_hint))\n    return _gate_and_select(\n        candidates, kalman_pred, gate_radius, velocity=velocity,\n        frame_width=frame_width, is_pre_contact=is_pre_contact,\n    )\n\n\n# Backward-compat alias used in _detect_one\ndef _detect_ball_bg_sub(\n    gray: np.ndarray,\n    bg_model: np.ndarray,                # kept in signature for compat \u2014 not used\n    size_hint: tuple[int, int] = BALL_SIZE_HINT,\n    threshold: int = 25,\n) -> Optional[tuple[float, float, float]]:\n    \"\"\"Compat stub \u2014 callers inside track_ball now use detect_ball() directly.\"\"\"\n    return None   # track_ball bypasses this via detect_ball()\n\n\n# ---------------------------------------------------------------------------\n# Trajectory quality validation\n# ---------------------------------------------------------------------------\n\ndef _validate_trajectory(dets: list[dict], fps: float) -> float:\n    \"\"\"\n    Fit a parabola to post-contact detections and return the R\u00b2 of the\n    vertical (y) component.\n\n    A real ball trajectory should follow y(t) \u2248 a\u00b7t\u00b2 + b\u00b7t + c closely.\n    Background noise produces near-random scatter \u2192 R\u00b2 \u2248 0.\n    A clean ball track \u2192 R\u00b2 \u2265 0.70.\n\n    Returns 0.0 if fewer than MIN_DETECTIONS_FOR_ANGLE points are present.\n    \"\"\"\n    real_dets = [d for d in dets if not d.get(\"predicted\", False)]\n    if len(real_dets) < MIN_DETECTIONS_FOR_ANGLE:\n        return 0.0\n    ts = np.arange(len(real_dets), dtype=float) / fps\n    ys = np.array([d[\"cy\"] for d in real_dets], dtype=float)\n    try:\n        poly   = np.polyfit(ts, ys, 2)\n        y_pred = np.polyval(poly, ts)\n        ss_res = float(np.sum((ys - y_pred) ** 2))\n        ss_tot = float(np.sum((ys - np.mean(ys)) ** 2))\n        return round(max(0.0, 1.0 - ss_res / (ss_tot + 1e-9)), 3)\n    except Exception:\n        return 0.0\n\n\n# ---------------------------------------------------------------------------\n# Contact frame detection (wrist-velocity heuristic)\n# NOTE: This is a lightweight proxy used only here to establish the\n#       contact_frame so the ball-tracking window is correctly centred.\n#       The authoritative contact_frame comes from mediapipe_processing.py;\n#       both scripts write their own estimate to JSON and mediapipe wins.\n# ---------------------------------------------------------------------------\n\ndef _estimate_contact_frame_from_motion(\n    video_path: Path,\n    fps: float,\n    search_start: int = 0,\n    search_end: Optional[int] = None,\n) -> int:\n    \"\"\"\n    Estimate the contact frame using dense optical flow (frame-difference\n    energy), which works without MediaPipe and runs in the YOLO environment.\n\n    Strategy: the frame with the largest sudden *drop* in motion energy after\n    a high-energy peak is the contact frame (the bat decelerates on contact).\n\n    Falls back to the middle of the video if the signal is ambiguous.\n    \"\"\"\n    cap = cv2.VideoCapture(str(video_path))\n    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))\n    end   = search_end or total\n\n    energies: list[tuple[int, float]] = []\n    prev_gray: Optional[np.ndarray] = None\n\n    # Sample every 2 frames for speed\n    for fi in range(search_start, end, 2):\n        cap.set(cv2.CAP_PROP_POS_FRAMES, fi)\n        ret, frame = cap.read()\n        if not ret:\n            break\n        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)\n        gray = cv2.GaussianBlur(gray, (5, 5), 0)\n        if prev_gray is not None:\n            diff   = cv2.absdiff(gray, prev_gray)\n            energy = float(np.mean(diff))\n            energies.append((fi, energy))\n        prev_gray = gray\n\n    cap.release()\n\n    if len(energies) < 10:\n        return total // 2   # fallback: middle of video\n\n    idxs = np.array([e[0] for e in energies], dtype=int)\n    vals = np.array([e[1] for e in energies], dtype=float)\n\n    # Smooth to reduce noise\n    from scipy.signal import savgol_filter\n    win = min(15, len(vals) - (1 - len(vals) % 2))\n    win = max(win if win % 2 == 1 else win - 1, 5)\n    if len(vals) > win:\n        smooth = savgol_filter(vals, win, 3)\n    else:\n        smooth = vals\n\n    # Gate peak search to 30%-85% of clip to exclude load/follow-through phases\n    n          = len(smooth)\n    gate_lo    = max(0,   int(n * 0.30))\n    gate_hi    = min(n-1, int(n * 0.85))\n    gated      = smooth.copy()\n    gated[:gate_lo]     = 0.0\n    gated[gate_hi + 1:] = 0.0\n\n    peak_i    = int(np.argmax(gated))\n    contact_i = peak_i\n    threshold = smooth[peak_i] * 0.70\n    post = smooth[peak_i:]\n    for i in range(1, len(post) - 1):\n        if post[i] < post[i-1] and post[i] < post[i+1] and post[i] < threshold:\n            contact_i = peak_i + i\n            break\n\n    return int(idxs[contact_i])\n\n\n# ---------------------------------------------------------------------------\n# Ball trajectory tracking\n# ---------------------------------------------------------------------------\n\ndef _fit_launch_angle(\n    centroids: list[tuple[float, float]],\n    fps: float,\n    pitcher_side: Optional[str] = PITCHER_SIDE,\n) -> Optional[float]:\n    \"\"\"\n    Fit a parabola to the list of (cx, cy) pixel centroids and extract the\n    initial launch angle.\n\n    Image-space convention: y increases downward.\n    We flip vy sign so that upward motion = positive angle (world convention).\n\n    Pitcher-side correction:\n      PITCHER_SIDE = \"R\" \u2192 ball comes from LEFT, post-contact vx is positive\n                           (ball moves right in image).  atan2 is correct.\n      PITCHER_SIDE = \"L\" \u2192 ball comes from RIGHT, post-contact vx is negative\n                           (ball moves left in image).  We use abs(vx) so the\n                           horizontal component is always positive, and the sign\n                           comes only from vy (vertical launch).\n\n    Returns degrees in approximately [-90, +90], or None if < 5 points.\n    \"\"\"\n    if len(centroids) < MIN_DETECTIONS_FOR_ANGLE:\n        return None\n    ts = np.arange(len(centroids), dtype=float) / fps\n    xs = np.array([c[0] for c in centroids])\n    ys = np.array([c[1] for c in centroids])\n    try:\n        vx_raw   = float(np.mean(np.diff(xs)) * fps)\n        poly_y   = np.polyfit(ts, ys, 2)\n        vy_pixel = float(np.polyval(np.polyder(poly_y), 0.0))\n        vy       = -vy_pixel   # image y-down \u2192 world y-up\n\n        # Validate horizontal direction vs pitcher side\n        expected_sign = _post_contact_direction(pitcher_side)\n        if expected_sign is not None and abs(vx_raw) > 2.0:\n            if (vx_raw * expected_sign) < 0:\n                # vx points the wrong way \u2014 the parabola fit latched onto\n                # pre-contact frames or a background blob.  Flip so angle\n                # reflects the true post-contact direction.\n                log.warning(\n                    \"  vx sign (%.1f) disagrees with pitcher_side=%s \u2014 correcting\",\n                    vx_raw, pitcher_side,\n                )\n                vx_raw = -vx_raw\n\n        return round(math.degrees(math.atan2(vy, abs(vx_raw))), 2)\n    except Exception:\n        return None\n\n\ndef track_ball(\n    video_path: Path,\n    contact_frame: int,\n    fps: float,\n    detector: Optional[YOLODetector],\n    pitcher_side: Optional[str] = PITCHER_SIDE,\n) -> dict:\n    \"\"\"\n    YOLO + temporal-diff + Kalman ball tracking.\n\n    All frame-based detection uses frame-to-frame differencing (current vs\n    previous frame), not a static background model. This keeps the batter's\n    moving body out of the diff image, leaving only fast-moving objects (the\n    ball) as strong candidates.\n\n    Pre-contact : sequential forward scan, maintaining prev_gray at every step.\n    Post-contact: continues the same prev_gray chain for temporal consistency.\n\n    Gate radius shrinks from 60px (no prior) to 30px (6+ confirmed detections).\n\n    Pitcher-side constraints:\n      - Candidates on the wrong frame-side are score-penalised.\n      - Kalman velocity is checked; if it strongly points the wrong way the\n        detection is rejected to prevent the tracker drifting to crowd blobs.\n    \"\"\"\n    cap = cv2.VideoCapture(str(video_path))\n\n    # Video dimensions for side-bias scoring\n    frame_width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))\n    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))  # noqa: F841 (kept for completeness)\n\n    # Pre-contact: ball must come FROM the pitcher's release side.\n    # Post-contact: ball travels AWAY from that side.\n    pre_origin = _pitcher_origin_x_range(frame_width)\n    if pitcher_side is not None and pre_origin is not None:\n        log.info(\n            \"  Pitcher-side constraint: pitcher=%s  |  pre-contact origin x\u2208[%d,%d]  \"\n            \"|  post-contact expected vx-sign=%+d\",\n            pitcher_side, pre_origin[0], pre_origin[1],\n            _post_contact_direction(pitcher_side) or 0,\n        )\n\n    def _gate_r(n_confirmed: int) -> float:\n        if n_confirmed == 0: return 60.0\n        if n_confirmed < 3:  return 50.0\n        if n_confirmed < 6:  return 40.0\n        return 30.0\n\n    def _detect_one_temporal(\n        bgr: np.ndarray,\n        prev_gray: np.ndarray,\n        kalman_pred: Optional[tuple[float, float]],\n        gate_radius: float,\n        velocity: Optional[tuple[float, float]] = None,\n        last_confirmed: Optional[tuple[float, float]] = None,\n        is_pre: bool = True,\n    ) -> Optional[tuple[float, float, float]]:\n        gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)\n\n        # \u2500\u2500 Pitcher-side velocity guard \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n        # If the Kalman velocity is firmly headed the WRONG horizontal way,\n        # skip this frame's detection to avoid the tracker latching onto a\n        # crowd blob moving in the opposite direction to the ball.\n        if not _enforce_vx_direction(velocity, is_pre_contact=is_pre):\n            log.debug(\n                \"  vx direction guard triggered (phase=%s, vx=%.1f)\",\n                \"pre\" if is_pre else \"post\", velocity[0] if velocity else 0,\n            )\n            return None\n\n        # YOLO path \u2014 gated so COCO crowd detections are rejected.\n        # Gate tightened from 1.5\u00d7 to 1.0\u00d7 to reduce false positives from\n        # crowd \"sports ball\" detections that survive at looser thresholds.\n        if detector is not None:\n            try:\n                ball, _ = detector.detect(bgr)\n                if ball is not None:\n                    cx, cy, r = ball\n                    yolo_ok = (\n                        kalman_pred is None or\n                        math.sqrt((cx - kalman_pred[0])**2 +\n                                  (cy - kalman_pred[1])**2) <= gate_radius\n                    )\n                    if yolo_ok:\n                        # Speed filter: reject near-static YOLO hits (background)\n                        if last_confirmed is not None:\n                            dist = math.sqrt((cx - last_confirmed[0])**2 +\n                                             (cy - last_confirmed[1])**2)\n                            if dist < MIN_BALL_SPEED_PX_PER_FRAME:\n                                yolo_ok = False\n                    if yolo_ok:\n                        # Pitcher-side origin filter for YOLO detections\n                        if pitcher_side is not None and is_pre and pre_origin is not None:\n                            # Hard-reject YOLO pre-contact hits on the WRONG side\n                            # only when the tracker has no prior at all (first few frames)\n                            if kalman_pred is None:\n                                bias = _side_bias_score(cx, frame_width, is_pre_contact=True)\n                                if bias < 1.0:\n                                    # Wrong-side hit with no prior \u2192 skip\n                                    yolo_ok = False\n                    if yolo_ok:\n                        return ball\n            except Exception as e:\n                if \"numpy\" not in str(e).lower() and \"not available\" not in str(e).lower():\n                    log.debug(\"YOLO error: %s\", e)\n\n        # Temporal diff detector \u2014 always runs, no bg model needed\n        candidate = detect_ball(\n            gray, prev_gray, BALL_SIZE_HINT,\n            kalman_pred=kalman_pred,\n            gate_radius=gate_radius,\n            velocity=velocity,\n            frame_width=frame_width,\n            is_pre_contact=is_pre,\n        )\n\n        # Speed filter on temporal-diff result too\n        if candidate is not None and last_confirmed is not None:\n            cx, cy, _ = candidate\n            dist = math.sqrt((cx - last_confirmed[0])**2 +\n                             (cy - last_confirmed[1])**2)\n            if dist < MIN_BALL_SPEED_PX_PER_FRAME:\n                candidate = None\n\n        return candidate\n\n    bat_boxes: dict[str, list[float]] = {}\n\n    # \u2500\u2500 Pre-contact: sequential scan so prev_gray is always one frame back \u2500\u2500\n    pre_kf   = BallKalmanFilter(fps)\n    pre_dets: list[dict] = []\n    n_confirmed_pre = 0\n    last_confirmed_pre: Optional[tuple[float, float]] = None\n\n    scan_start = max(0, contact_frame - PRE_SEARCH_FRAMES)\n\n    # Seed prev_gray from one frame before the scan window starts\n    cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, scan_start - 1))\n    ret, seed = cap.read()\n    prev_gray: np.ndarray = (\n        cv2.cvtColor(seed, cv2.COLOR_BGR2GRAY) if ret\n        else np.zeros((1080, 1920), dtype=np.uint8)\n    )\n\n    cap.set(cv2.CAP_PROP_POS_FRAMES, scan_start)\n    for fi in range(scan_start, contact_frame):\n        ret, frame = cap.read()\n        if not ret:\n            break\n\n        if detector is not None:\n            try:\n                _, bat = detector.detect(frame)\n                if bat is not None:\n                    bat_boxes[str(fi)] = list(bat)\n            except Exception:\n                pass\n\n        pred = pre_kf.predict()\n        vel  = pre_kf.velocity()\n        gate = _gate_r(n_confirmed_pre)\n\n        ball = _detect_one_temporal(frame, prev_gray, pred, gate,\n                                    velocity=vel,\n                                    last_confirmed=last_confirmed_pre,\n                                    is_pre=True)\n        if ball is not None:\n            cx_raw, cy_raw, r = ball\n            cx, cy = pre_kf.update(cx_raw, cy_raw)\n            pre_dets.append({\"frame\": fi, \"cx\": cx, \"cy\": cy, \"r\": r})\n            n_confirmed_pre += 1\n            last_confirmed_pre = (cx, cy)\n        elif pred is not None and pre_dets:\n            last_fi = pre_dets[-1][\"frame\"]\n            if fi - last_fi <= 4:\n                px, py = pred\n                pre_dets.append({\"frame\": fi, \"cx\": px, \"cy\": py,\n                                 \"r\": pre_dets[-1][\"r\"], \"predicted\": True})\n\n        prev_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)\n\n    # \u2500\u2500 Post-contact: continue prev_gray chain \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    post_kf          = BallKalmanFilter(fps)\n    post_dets: list[dict] = []\n    ball_contact_px: Optional[list[float]] = None\n    consecutive_misses = 0\n    n_confirmed_post   = 0\n    last_confirmed_post: Optional[tuple[float, float]] = None\n\n    cap.set(cv2.CAP_PROP_POS_FRAMES, contact_frame)\n    for _ in range(POST_SEARCH_FRAMES):\n        ret, frame = cap.read()\n        if not ret:\n            break\n        fi = int(cap.get(cv2.CAP_PROP_POS_FRAMES)) - 1\n\n        if detector is not None:\n            try:\n                _, bat = detector.detect(frame)\n                if bat is not None:\n                    bat_boxes[str(fi)] = list(bat)\n            except Exception:\n                pass\n\n        pred = post_kf.predict()\n        vel  = post_kf.velocity()\n        gate = _gate_r(n_confirmed_post) * 1.4   # slightly wider \u2014 ball moves fast\n\n        ball = _detect_one_temporal(frame, prev_gray, pred, gate,\n                                    velocity=vel,\n                                    last_confirmed=last_confirmed_post,\n                                    is_pre=False)\n\n        if ball is not None:\n            cx_raw, cy_raw, r = ball\n            cx, cy = post_kf.update(cx_raw, cy_raw)\n            post_dets.append({\"frame\": fi, \"cx\": cx, \"cy\": cy, \"r\": r})\n            consecutive_misses = 0\n            n_confirmed_post  += 1\n            last_confirmed_post = (cx, cy)\n            if ball_contact_px is None:\n                ball_contact_px = [cx, cy]\n        else:\n            consecutive_misses += 1\n            if pred is not None and consecutive_misses <= MAX_CONSECUTIVE_MISSES \\\n                    and post_dets:\n                px, py = pred\n                last_r = post_dets[-1][\"r\"]\n                post_dets.append({\"frame\": fi, \"cx\": px, \"cy\": py,\n                                  \"r\": last_r, \"predicted\": True})\n            elif consecutive_misses > MAX_CONSECUTIVE_MISSES:\n                break\n\n        prev_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)\n\n    cap.release()\n\n    # \u2500\u2500 Trajectory quality validation \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    # Fit a parabola to post-contact detections.  If R\u00b2 is below the threshold\n    # the tracker latched onto background noise rather than the ball, so we\n    # null the angle and contact pixel to prevent bad annotations downstream.\n    traj_r2    = _validate_trajectory(post_dets, fps)\n    post_centroids = [(d[\"cx\"], d[\"cy\"]) for d in post_dets]\n\n    if traj_r2 >= TRAJECTORY_MIN_R2:\n        ball_angle = _fit_launch_angle(post_centroids, fps, pitcher_side=pitcher_side)\n        log.info(\"  Trajectory R\u00b2=%.3f \u2014 ball track accepted\", traj_r2)\n    else:\n        ball_angle     = None\n        ball_contact_px = None   # don't annotate with a bad contact point\n        log.warning(\n            \"  Trajectory R\u00b2=%.3f < %.2f \u2014 track rejected as noise \"\n            \"(ball_angle and ball_contact_px nulled)\",\n            traj_r2, TRAJECTORY_MIN_R2,\n        )\n\n    return {\n        \"post_contact\":    post_dets,\n        \"pre_contact\":     pre_dets,\n        \"bat_boxes\":       bat_boxes,\n        \"ball_angle\":      ball_angle,\n        \"ball_contact_px\": ball_contact_px,\n    }\n\n\n# ---------------------------------------------------------------------------\n# Per-video processing\n# ---------------------------------------------------------------------------\n\ndef process_video(video_path: Path, detector: Optional[YOLODetector]) -> dict:\n    \"\"\"\n    Full YOLO pipeline for one video.\n\n    1. Read video metadata.\n    2. Estimate contact frame via optical-flow energy drop.\n    3. Track ball pre/post contact with YOLO + Kalman (or bg-sub if YOLO unavailable).\n    4. Return the complete result dict (later written to JSON).\n    \"\"\"\n    cap = cv2.VideoCapture(str(video_path))\n    if not cap.isOpened():\n        raise IOError(f\"Cannot open video: {video_path}\")\n    fps    = cap.get(cv2.CAP_PROP_FPS) or 60.0\n    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))\n    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))\n    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))\n    cap.release()\n\n    log.info(\"  Video: %dx%d @ %.0ffps, %d frames\", width, height, fps, total)\n\n    # Estimate contact frame (motion energy heuristic)\n    contact_frame = _estimate_contact_frame_from_motion(video_path, fps)\n    contact_time  = round(contact_frame / fps, 3)\n    log.info(\"  Estimated contact frame: %d (%.3fs)\", contact_frame, contact_time)\n\n    # Ball + bat tracking\n    tracking = track_ball(video_path, contact_frame, fps, detector,\n                          pitcher_side=PITCHER_SIDE)\n\n    return {\n        \"video_path\":      str(video_path),\n        \"fps\":             fps,\n        \"width\":           width,\n        \"height\":          height,\n        \"contact_frame\":   contact_frame,\n        \"contact_time_s\":  contact_time,\n        \"pitcher_side\":    PITCHER_SIDE,          # \u2190 stored for mediapipe stage\n        \"ball_angle\":      tracking[\"ball_angle\"],\n        \"ball_contact_px\": tracking[\"ball_contact_px\"],\n        \"post_contact\":    tracking[\"post_contact\"],\n        \"pre_contact\":     tracking[\"pre_contact\"],\n        \"bat_boxes\":       tracking[\"bat_boxes\"],\n    }\n\n\ndef sidecar_path(video_path: Path) -> Path:\n    \"\"\"Return the path for the JSON sidecar file next to a video.\"\"\"\n    return video_path.with_suffix(\".yolo.json\")\n\n\n# ---------------------------------------------------------------------------\n# Main\n# ---------------------------------------------------------------------------\n\ndef _check_numpy_compat() -> bool:\n    \"\"\"\n    Ultralytics silently breaks when numpy is present but the wrong ABI\n    version is loaded. This check catches the issue before the first\n    inference call and prints an actionable fix instead of a cryptic crash.\n\n    Returns True if numpy is compatible, False otherwise.\n    \"\"\"\n    try:\n        import numpy as _np\n        # Force a real array operation \u2014 this is what ultralytics does internally\n        _ = _np.array([1.0, 2.0], dtype=_np.float32) * 2\n        return True\n    except Exception as e:\n        log.error(\"NumPy compatibility check failed: %s\", e)\n        log.error(\n            \"Fix: pip install 'numpy>=1.23,<2.1'  (ultralytics requires numpy \"\n            \"compiled against a compatible ABI \u2014 reinstalling usually resolves this)\"\n        )\n        return False\n\n\ndef main() -> None:\n    if not VIDEO_ROOT.exists():\n        log.error(\"VIDEO_ROOT does not exist: %s\", VIDEO_ROOT)\n        sys.exit(1)\n\n    # Verify numpy is usable before doing any work\n    numpy_ok = _check_numpy_compat()\n    if not numpy_ok:\n        log.warning(\n            \"NumPy is broken in this environment. \"\n            \"YOLO will be disabled and all videos will use background subtraction only.\"\n        )\n\n    # Find all videos\n    videos = sorted(\n        p for p in VIDEO_ROOT.rglob(\"*\")\n        if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS\n        and \"_annotated\" not in p.stem   # skip already-annotated clips\n    )\n    if not videos:\n        log.error(\"No videos found in %s\", VIDEO_ROOT)\n        sys.exit(1)\n\n    log.info(\"Found %d video(s)\", len(videos))\n\n    # Load YOLO \u2014 disabled automatically if numpy is broken\n    detector: Optional[YOLODetector] = None\n    if numpy_ok:\n        try:\n            detector = YOLODetector(YOLO_WEIGHTS, conf=CONF_THRESHOLD)\n        except Exception as e:\n            log.warning(\"YOLO failed to load (%s) \u2014 falling back to background subtraction.\", e)\n\n    if detector is None:\n        log.info(\"Running in background-subtraction-only mode (no YOLO).\")\n\n    success, failed = 0, []\n    for i, vp in enumerate(videos, 1):\n        out_path = sidecar_path(vp)\n        if out_path.exists():\n            log.info(\"[%d/%d] %s \u2014 sidecar exists, skipping\", i, len(videos), vp.name)\n            success += 1\n            continue\n\n        log.info(\"[%d/%d] %s\", i, len(videos), vp.name)\n        try:\n            result = process_video(vp, detector)\n            with open(out_path, \"w\") as f:\n                json.dump(result, f, indent=2)\n            log.info(\"  \u2192 %s\", out_path.name)\n            success += 1\n        except Exception as e:\n            log.warning(\"  FAILED: %s\", e)\n            failed.append((vp.name, str(e)))\n\n    print(f\"\\n\u2714 {success}/{len(videos)} videos processed \u2192 JSON sidecars in {VIDEO_ROOT}\")\n    if failed:\n        print(f\"\u2717 Failed ({len(failed)}):\")\n        for name, err in failed:\n            print(f\"    {name}: {err}\")\n\n    print(\"\\nNext step: run mediapipe_processing.py in the MediaPipe environment.\")\n\n\nif __name__ == \"__main__\":\n    main()"
with open('/content/yolo_processing.py','w') as f: f.write(yolo_src)
print('yolo_processing.py written')

yolo_processing.py written


In [ ]:
mp_src = "\"\"\"\nmediapipe_processing.py\n=======================\nStage 2 of the two-stage baseball swing analysis pipeline.\n\nResponsibility\n--------------\n- Load each swing video + its .yolo.json sidecar (written by yolo_processing.py)\n- Run MediaPipe BlazePose to extract 3-D body landmarks for every frame\n- Compute the authoritative contact frame from wrist deceleration (overrides\n  the optical-flow estimate from Stage 1)\n- Compute launch angle from the bat vector (pose-based) and blend with the\n  ball-tracked angle from the sidecar\n- Classify contact zone (9-zone strike grid) and predict landing zone\n- Write per-video JSON results and a summary batch_results.csv\n- Optionally render fully annotated videos (proof-check mode)\n\nInput: .yolo.json sidecars written by yolo_processing.py (one per video)\nOutput:\n  trimmed_videos/outputs/<stem>_result.json    \u2014 per-video metrics\n  trimmed_videos/outputs/batch_results.csv     \u2014 all videos in one table\n  trimmed_videos/<stem>_annotated.mp4          \u2014 (proof-check mode only)\n\nExpected sidecar format (.yolo.json)\n-------------------------------------\n{\n  \"video_path\":    \"/abs/path/to/video.mp4\",\n  \"fps\":           60.0,\n  \"width\":         1722,\n  \"height\":        1080,\n  \"contact_frame\": 238,\n  \"contact_time_s\": 3.967,\n  \"ball_angle\":    14.3,\n  \"ball_contact_px\": [841, 512],\n  \"post_contact\":  [{\"frame\": 238, \"cx\": 841.2, \"cy\": 512.7, \"r\": 9.1}, ...],\n  \"pre_contact\":   [{\"frame\": 220, \"cx\": 310.5, \"cy\": 490.1, \"r\": 8.4}, ...],\n  \"bat_boxes\":     {\"235\": [1020.1, 380.5, 1180.3, 560.2], ...}\n}\n\nEnvironment\n-----------\n  conda create -n mediapipe_env python=3.10\n  pip install \"numpy<2\" mediapipe opencv-python scipy\n\nModel download (run once)\n-------------------------\n  python mediapipe_processing.py --download-model\n\nRun\n---\n  python mediapipe_processing.py\n\"\"\"\n\nfrom __future__ import annotations\n\nimport csv\nimport json\nimport logging\nimport math\nimport sys\nimport urllib.request\nfrom dataclasses import asdict, dataclass, field\nfrom pathlib import Path\nfrom typing import Optional\n\nimport cv2\nimport numpy as np\nfrom scipy.signal import savgol_filter\n\nlogging.basicConfig(\n    level=logging.INFO,\n    format=\"%(asctime)s [%(levelname)s] %(message)s\",\n)\nlog = logging.getLogger(__name__)\n\n# ---------------------------------------------------------------------------\n# Configuration  \u2190 edit these, then run: python mediapipe_processing.py\n# ---------------------------------------------------------------------------\n\nVIDEO_ROOT   = Path(__file__).parent / \"trimmed_videos\"\nOUTPUT_DIR   = VIDEO_ROOT / \"outputs\"\nBATCH_CSV    = OUTPUT_DIR / \"batch_results.csv\"\nMODEL_PATH   = Path(__file__).parent / \"models\" / \"pose_landmarker_lite.task\"\nMODEL_URL    = (\n    \"https://storage.googleapis.com/mediapipe-models/\"\n    \"pose_landmarker/pose_landmarker_lite/float16/latest/\"\n    \"pose_landmarker_lite.task\"\n)\n\n# Proof-check: process only the first N videos and render annotated clips\nPROOF_CHECK   = True    # \u2190 flip to False for full batch run\nPROOF_CHECK_N = 10\n\nSTRIDE            = 6       # frame stride for pose estimation — 6 = 10fps @60fps (~3x faster than 2)\nYOLO_ANCHOR_WINDOW = 54       # ±54 frames (±0.9s @60fps) around YOLO contact estimate for pose window\nENABLE_BALL_BLEND = True    # blend ball-tracked angle from YOLO sidecar into final angle\nVERBOSE           = True\n\n# Maximum pixel distance between ball_contact_px (from YOLO sidecar) and the\n# wrist midpoint from pose landmarks. If the YOLO-derived contact pixel is\n# further away than this, we assume it latched onto background motion and fall\n# back to the wrist midpoint for the contact annotation.\nMAX_CONTACT_WRIST_DIST_PX = 250\n\nVIDEO_EXTENSIONS = {\".mp4\", \".mov\", \".avi\", \".mkv\"}\n\n# ---------------------------------------------------------------------------\n# MediaPipe landmark indices (BlazePose 33-point model)\n# ---------------------------------------------------------------------------\nLM = {\n    \"l_shoulder\": 11, \"r_shoulder\": 12,\n    \"l_elbow\":    13, \"r_elbow\":    14,\n    \"l_wrist\":    15, \"r_wrist\":    16,\n    \"l_hip\":      23, \"r_hip\":      24,\n    \"l_ankle\":    27, \"r_ankle\":    28,\n    \"l_knee\":     25, \"r_knee\":     26,\n}\n\n# Launch angle \u2192 field zone lookup\nANGLE_ZONES = [\n    (-90, -10, \"ground_ball\",     \"Infield / foul\"),\n    (-10,   5, \"hard_liner\",      \"Hard liner / gap\"),\n    (  5,  15, \"line_drive\",      \"Line drive (ideal)\"),\n    ( 15,  25, \"solid_fly\",       \"Solid fly ball\"),\n    ( 25,  35, \"home_run_window\", \"Home run window\"),\n    ( 35,  50, \"high_fly\",        \"High fly / shallow OF\"),\n    ( 50,  90, \"popup\",           \"Pop-up\"),\n]\n\nCP_VERTICAL   = {\"high\": 0.65, \"mid_hi\": 0.45, \"mid_lo\": 0.25, \"low\": 0.0}\nCP_HORIZONTAL = {\"inside\": 0.33, \"middle\": 0.66}\n\n# Annotation colours (BGR)\n_COL_SKEL_PRE    = (100, 220, 100)   # green skeleton before contact\n_COL_SKEL_POST   = (100, 180, 255)   # blue skeleton after contact\n_COL_CONTACT     = (0,    80, 255)   # red-orange contact rings\n_COL_BALL        = (0,   220, 255)   # yellow ball dot\n_COL_RAY         = (255,  80,  80)   # cyan angle ray\n_COL_HUD_BG      = (20,   20,  20)   # dark HUD panel\n_COL_BAT_BOX     = (60,  220, 255)   # orange bat bounding box\n\n\n# ---------------------------------------------------------------------------\n# Data classes\n# ---------------------------------------------------------------------------\n\n@dataclass\nclass Landmark3D:\n    x: float\n    y: float\n    z: float\n    visibility: float = 1.0\n\n    def as_array(self) -> np.ndarray:\n        return np.array([self.x, self.y, self.z], dtype=np.float64)\n\n\n@dataclass\nclass SwingResult:\n    \"\"\"One row in batch_results.csv / one per-video JSON.\"\"\"\n    video_path:          str\n    contact_frame:       int\n    contact_time_s:      float\n    launch_angle:        float          # final blended angle, degrees [-90, +90]\n    bat_vector:          list[float]    # normalised 3-D bat direction\n    contact_zone:        str            # e.g. \"Mid-In\"\n    landing_zone:        str            # e.g. \"home_run_window\"\n    outcome_description: str\n    confidence:          float          # 0\u20131 landmark visibility score\n    batter_handedness:   str            # \"R\" | \"L\" | \"unknown\"\n    # Ball-tracking fields (populated from YOLO sidecar when available)\n    ball_angle:          Optional[float] = None\n    blended_angle:       Optional[float] = None\n    ball_centroids:      list = field(default_factory=list)\n    ball_radii:          list = field(default_factory=list)\n    pre_contact_centroids: list = field(default_factory=list)\n    pre_contact_radii:   list = field(default_factory=list)\n    ball_contact_px:     Optional[list] = None\n    pitcher_side:        Optional[str]  = None   # \"L\", \"R\", or None\n\n\n# ---------------------------------------------------------------------------\n# Model helpers\n# ---------------------------------------------------------------------------\n\ndef download_model(dest: Path = MODEL_PATH) -> None:\n    \"\"\"Download the MediaPipe PoseLandmarker model if not already present.\"\"\"\n    if dest.exists() and dest.stat().st_size > 1_000_000:\n        log.info(\"Model already present: %s\", dest)\n        return\n    dest.parent.mkdir(parents=True, exist_ok=True)\n    log.info(\"Downloading PoseLandmarker model \u2192 %s \u2026\", dest)\n    urllib.request.urlretrieve(MODEL_URL, dest)\n    log.info(\"Download complete (%.1f MB)\", dest.stat().st_size / 1e6)\n\n\ndef load_pose_landmarker(model_path: Path = MODEL_PATH):\n    \"\"\"\n    Load a MediaPipe PoseLandmarker (Tasks API, mediapipe \u2265 0.10).\n\n    A fresh instance must be created per video because VIDEO running mode\n    requires monotonically increasing timestamps with no reset method.\n    \"\"\"\n    if not model_path.exists():\n        raise FileNotFoundError(\n            f\"Model not found at {model_path}. \"\n            \"Run: python mediapipe_processing.py --download-model\"\n        )\n    import mediapipe as mp\n    from mediapipe.tasks.python import vision\n    from mediapipe.tasks.python.core.base_options import BaseOptions\n\n    options = vision.PoseLandmarkerOptions(\n        base_options=BaseOptions(model_asset_path=str(model_path)),\n        running_mode=vision.RunningMode.VIDEO,\n        num_poses=1,\n        min_pose_detection_confidence=0.5,\n        min_pose_presence_confidence=0.5,\n        min_tracking_confidence=0.5,\n    )\n    return vision.PoseLandmarker.create_from_options(options)\n\n\n# ---------------------------------------------------------------------------\n# Stage 1: Video ingestion\n# ---------------------------------------------------------------------------\n\ndef open_video(path: Path) -> tuple[cv2.VideoCapture, dict]:\n    cap = cv2.VideoCapture(str(path))\n    if not cap.isOpened():\n        raise IOError(f\"Cannot open video: {path}\")\n    return cap, {\n        \"fps\":          cap.get(cv2.CAP_PROP_FPS) or 60.0,\n        \"total_frames\": int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),\n        \"width\":        int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),\n        \"height\":       int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),\n    }\n\n\n# ---------------------------------------------------------------------------\n# Stage 2: Pose estimation (MediaPipe BlazePose)\n# ---------------------------------------------------------------------------\n\ndef _pad_to_square(frame: np.ndarray) -> tuple[np.ndarray, int, int]:\n    \"\"\"\n    Letterbox to square with black padding.\n    Fixes MediaPipe's NORM_RECT warning on 16:9 frames and prevents\n    coordinate distortion in landmark normalisation.\n    \"\"\"\n    h, w = frame.shape[:2]\n    size     = max(h, w)\n    pad_top  = (size - h) // 2\n    pad_left = (size - w) // 2\n    padded   = cv2.copyMakeBorder(\n        frame, pad_top, size - h - pad_top,\n        pad_left, size - w - pad_left,\n        cv2.BORDER_CONSTANT, value=(0, 0, 0),\n    )\n    return padded, pad_top, pad_left\n\n\ndef _unpad_landmarks(lms_norm, orig_h, orig_w, pad_top, pad_left) -> list[Landmark3D]:\n    \"\"\"\n    Map landmarks from padded-square normalised space back to\n    original-frame normalised space.\n    \"\"\"\n    size = max(orig_h, orig_w)\n    out  = []\n    for lm in lms_norm:\n        px = lm.x * size - pad_left\n        py = lm.y * size - pad_top\n        out.append(Landmark3D(px / orig_w, py / orig_h, lm.z, lm.visibility))\n    return out\n\n\ndef run_pose_on_frames(\n    video_path: Path,\n    landmarker,\n    stride: int = 1,\n    frame_range: tuple[int, int] | None = None,\n) -> list[tuple[int, list[Landmark3D] | None]]:\n    \"\"\"\n    Run PoseLandmarker on every Nth frame of the video.\n\n    frame_range: optional (start, end) inclusive — only process frames in this\n    window. Used to restrict pose estimation to a YOLO-anchored window around\n    the estimated contact frame, cutting 60-70% of frames processed.\n\n    Returns list of (frame_idx, landmarks_or_None).\n    Each landmark list has 66 entries:\n      [0..32]  world landmarks (metric, hip-centred)\n      [33..65] image landmarks (normalised, un-padded)\n    \"\"\"\n    import mediapipe as mp\n\n    cap, meta = open_video(video_path)\n    fps, orig_w, orig_h = meta[\"fps\"], meta[\"width\"], meta[\"height\"]\n    total = meta[\"total_frames\"]\n    results: list[tuple[int, list[Landmark3D] | None]] = []\n\n    # Determine frame window\n    if frame_range is not None:\n        win_start = max(0, frame_range[0])\n        win_end   = min(total - 1, frame_range[1])\n    else:\n        win_start, win_end = 0, total - 1\n\n    cap.set(cv2.CAP_PROP_POS_FRAMES, win_start)\n    idx = win_start\n    while idx <= win_end:\n        ret, frame = cap.read()\n        if not ret:\n            break\n        if (idx - win_start) % stride == 0:\n            padded, pad_top, pad_left = _pad_to_square(frame)\n            rgb       = cv2.cvtColor(padded, cv2.COLOR_BGR2RGB)\n            mp_image  = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)\n            ts_ms     = int((idx / fps) * 1000)\n            pose_res  = landmarker.detect_for_video(mp_image, ts_ms)\n\n            if pose_res.pose_world_landmarks and pose_res.pose_landmarks:\n                world_lms = [\n                    Landmark3D(lm.x, lm.y, lm.z, lm.visibility)\n                    for lm in pose_res.pose_world_landmarks[0]\n                ]\n                image_lms = _unpad_landmarks(\n                    pose_res.pose_landmarks[0], orig_h, orig_w, pad_top, pad_left\n                )\n                results.append((idx, world_lms + image_lms))\n            else:\n                results.append((idx, None))\n        idx += 1\n\n    cap.release()\n    return results\n\n\n# ---------------------------------------------------------------------------\n# Stage 3A: Contact frame detection (wrist deceleration)\n# ---------------------------------------------------------------------------\n\ndef compute_wrist_velocity(\n    frame_results: list[tuple[int, list[Landmark3D] | None]],\n    fps: float,\n) -> list[tuple[int, float]]:\n    \"\"\"Bilateral average wrist speed (world units/s) per frame.\"\"\"\n    velocities: list[tuple[int, float]] = []\n    prev_mid = None\n    prev_fi  = None\n\n    for fi, lms in frame_results:\n        if lms is None:\n            prev_mid = None\n            continue\n        lw  = lms[LM[\"l_wrist\"]].as_array()\n        rw  = lms[LM[\"r_wrist\"]].as_array()\n        mid = (lw + rw) / 2.0\n        if prev_mid is not None and prev_fi is not None:\n            dt = (fi - prev_fi) / fps\n            if dt > 0:\n                velocities.append((fi, float(np.linalg.norm(mid - prev_mid) / dt)))\n        prev_mid = mid\n        prev_fi  = fi\n\n    return velocities\n\n\ndef find_contact_frame(velocities: list[tuple[int, float]]) -> int:\n    \"\"\"\n    Two-stage contact detection:\n    A) Find the wrist-speed peak inside the MIDDLE 30%-85% of the clip.\n       Gating to this window prevents the load/stride phase (first 30%) and\n       the follow-through coast (last 15%) from being mistaken for the swing\n       peak \u2014 which caused early false contact frames (e.g. Bregman frame 12).\n    B) Find the first trough after the peak that drops below 70% of peak\n       speed \u2014 this is where the bat decelerates on ball contact.\n    \"\"\"\n    if not velocities:\n        raise ValueError(\"Empty velocity list.\")\n    idxs = np.array([v[0] for v in velocities], dtype=int)\n    vels = np.array([v[1] for v in velocities], dtype=np.float64)\n\n    win = min(15, len(vels))\n    win = max(win if win % 2 == 1 else win - 1, 5)\n    smooth = savgol_filter(vels, win, 3) if len(vels) > win else vels\n\n    # Gate peak search to 30%-85% of the clip to exclude load/follow-through\n    n          = len(smooth)\n    gate_lo    = max(0,   int(n * 0.30))\n    gate_hi    = min(n-1, int(n * 0.85))\n    gated_vels = smooth.copy()\n    gated_vels[:gate_lo]    = 0.0\n    gated_vels[gate_hi + 1:] = 0.0\n\n    peak_i    = int(np.argmax(gated_vels))\n    contact_i = peak_i\n    threshold = smooth[peak_i] * 0.70\n    for i in range(1, len(smooth) - peak_i - 1):\n        p = peak_i + i\n        if smooth[p] < smooth[p-1] and smooth[p] < smooth[p+1] and smooth[p] < threshold:\n            contact_i = p\n            break\n\n    return int(idxs[contact_i])\n\n\n# ---------------------------------------------------------------------------\n# Stage 4: Launch angle geometry\n# ---------------------------------------------------------------------------\n\ndef infer_handedness(lms: list[Landmark3D]) -> str:\n    \"\"\"\n    Shoulder Z-depth heuristic: the front shoulder has a more negative Z\n    (closer to camera) in MediaPipe world coords.\n    RHH: left shoulder closer \u2192 l_sh_z < r_sh_z.\n    \"\"\"\n    try:\n        l_sh_z = lms[LM[\"l_shoulder\"]].z\n        r_sh_z = lms[LM[\"r_shoulder\"]].z\n        l_hi_z = lms[LM[\"l_hip\"]].z\n        r_hi_z = lms[LM[\"r_hip\"]].z\n        shoulder_sig = \"R\" if l_sh_z < r_sh_z else \"L\"\n        hip_sig      = \"R\" if l_hi_z < r_hi_z else \"L\"\n        return shoulder_sig if shoulder_sig == hip_sig else shoulder_sig\n    except Exception:\n        return \"unknown\"\n\n\ndef compute_bat_vector(lms: list[Landmark3D], handedness: str) -> np.ndarray:\n    \"\"\"\n    Visibility-weighted average of both wrist\u2192elbow vectors.\n    Lead arm (front arm at contact) receives 2\u00d7 weight.\n    \"\"\"\n    if handedness == \"L\":\n        lead_w, lead_e   = \"r_wrist\", \"r_elbow\"\n        trail_w, trail_e = \"l_wrist\", \"l_elbow\"\n    else:\n        lead_w, lead_e   = \"l_wrist\", \"l_elbow\"\n        trail_w, trail_e = \"r_wrist\", \"r_elbow\"\n\n    def _vec(wk: str, ek: str) -> tuple[np.ndarray, float]:\n        v   = lms[LM[wk]].as_array() - lms[LM[ek]].as_array()\n        vis = min(lms[LM[wk]].visibility, lms[LM[ek]].visibility)\n        n   = np.linalg.norm(v)\n        return (v / n if n > 1e-6 else np.zeros(3)), vis\n\n    lv, lvis = _vec(lead_w,  lead_e)\n    tv, tvis = _vec(trail_w, trail_e)\n    if np.dot(lv, tv) < 0:\n        tv = -tv\n\n    wl, wt   = 2.0 * lvis, 1.0 * tvis\n    blended  = (wl * lv + wt * tv) / (wl + wt + 1e-9)\n    n        = np.linalg.norm(blended)\n    if n < 1e-6:\n        raise ValueError(\"Degenerate bat vector.\")\n    return blended / n\n\n\ndef compute_sagittal_normal(lms: list[Landmark3D]) -> np.ndarray:\n    l_sh = lms[LM[\"l_shoulder\"]].as_array()\n    r_sh = lms[LM[\"r_shoulder\"]].as_array()\n    l_hi = lms[LM[\"l_hip\"]].as_array()\n    r_hi = lms[LM[\"r_hip\"]].as_array()\n    trunk = ((l_sh + r_sh) / 2) - ((l_hi + r_hi) / 2)\n    lr    = r_sh - l_sh\n    n     = np.cross(trunk, lr)\n    norm  = np.linalg.norm(n)\n    return n / norm if norm > 1e-6 else np.array([0., 0., 1.])\n\n\ndef project_sagittal(v: np.ndarray, normal: np.ndarray) -> np.ndarray:\n    proj = v - np.dot(v, normal) * normal\n    n    = np.linalg.norm(proj)\n    return proj / n if n > 1e-6 else proj\n\n\ndef launch_angle_world(bat_proj: np.ndarray) -> float:\n    \"\"\"Signed angle from horizontal in world space (y-up). Range \u2248 [-90, +90].\"\"\"\n    return round(float(math.degrees(math.atan2(bat_proj[1], abs(bat_proj[0])))), 2)\n\n\ndef launch_angle_image(lms: list[Landmark3D], handedness: str) -> Optional[float]:\n    \"\"\"\n    2-D image-space launch angle from un-padded pixel landmarks.\n    Primary signal: X/Y pixels are accurate from a fixed broadcast camera;\n    Z depth is neural-net estimated and noisier.\n    \"\"\"\n    if len(lms) < 66:\n        return None\n    OFFSET = 33\n    if handedness == \"L\":\n        lw, le = OFFSET + LM[\"r_wrist\"], OFFSET + LM[\"r_elbow\"]\n        tw, te = OFFSET + LM[\"l_wrist\"], OFFSET + LM[\"l_elbow\"]\n    else:\n        lw, le = OFFSET + LM[\"l_wrist\"], OFFSET + LM[\"l_elbow\"]\n        tw, te = OFFSET + LM[\"r_wrist\"], OFFSET + LM[\"r_elbow\"]\n\n    lvis = min(lms[lw].visibility, lms[le].visibility)\n    tvis = min(lms[tw].visibility, lms[te].visibility)\n    if lvis < 0.4 and tvis < 0.4:\n        return None\n\n    def _v2d(wi, ei):\n        w  = np.array([lms[wi].x, lms[wi].y])\n        e  = np.array([lms[ei].x, lms[ei].y])\n        v  = w - e\n        n  = np.linalg.norm(v)\n        vis = min(lms[wi].visibility, lms[ei].visibility)\n        return (v / n if n > 1e-6 else np.zeros(2)), vis\n\n    lv, lv_vis = _v2d(lw, le)\n    tv, tv_vis = _v2d(tw, te)\n    if np.dot(lv, tv) < 0:\n        tv = -tv\n\n    b = (2 * lv_vis * lv + tv_vis * tv) / (2 * lv_vis + tv_vis + 1e-9)\n    return round(float(math.degrees(math.atan2(-b[1], abs(b[0])))), 2)\n\n\n# ---------------------------------------------------------------------------\n# Stage 5: Zone classification\n# ---------------------------------------------------------------------------\n\ndef angle_to_landing_zone(angle: float) -> tuple[str, str]:\n    for lo, hi, key, desc in ANGLE_ZONES:\n        if lo <= angle < hi:\n            return key, desc\n    return (\"popup\", \"Pop-up\") if angle >= 50 else (\"ground_ball\", \"Ground ball\")\n\n\ndef contact_point_zone(lms: list[Landmark3D], handedness: str) -> str:\n    \"\"\"9-zone strike zone classification using un-padded image-space landmarks.\"\"\"\n    try:\n        OFF = 33 if len(lms) >= 66 else 0\n        y   = lambda k: lms[OFF + LM[k]].y\n        x   = lambda k: lms[OFF + LM[k]].x\n\n        ankle_y    = (y(\"l_ankle\") + y(\"r_ankle\")) / 2\n        shoulder_y = (y(\"l_shoulder\") + y(\"r_shoulder\")) / 2\n        span       = abs(shoulder_y - ankle_y)\n        lead_w     = \"l_wrist\" if handedness == \"R\" else \"r_wrist\"\n        wrist_y    = y(lead_w)\n\n        lo, hi  = min(ankle_y, shoulder_y), max(ankle_y, shoulder_y)\n        ratio   = max(0.0, min(1.0, 1.0 - (wrist_y - lo) / (span + 1e-9)))\n\n        if   ratio > CP_VERTICAL[\"high\"]:   vert = \"High\"\n        elif ratio > CP_VERTICAL[\"mid_hi\"]: vert = \"Mid\"\n        elif ratio > CP_VERTICAL[\"mid_lo\"]: vert = \"Low-Mid\"\n        else:                               vert = \"Low\"\n\n        hc      = (x(\"l_hip\") + x(\"r_hip\")) / 2\n        pw      = abs(x(\"r_hip\") - x(\"l_hip\")) * 1.5\n        h_ratio = max(0.0, min(1.0, (x(lead_w) - (hc - pw / 2)) / (pw + 1e-9)))\n\n        if h_ratio < CP_HORIZONTAL[\"inside\"]:   horiz = \"In\"\n        elif h_ratio < CP_HORIZONTAL[\"middle\"]: horiz = \"Middle\"\n        else:                                   horiz = \"Out\"\n\n        if handedness == \"L\":\n            horiz = {\"In\": \"Out\", \"Out\": \"In\", \"Middle\": \"Middle\"}[horiz]\n\n        return f\"{vert}-{horiz}\"\n    except Exception:\n        return \"Unknown\"\n\n\ndef estimate_confidence(\n    frame_results: list[tuple[int, list[Landmark3D] | None]],\n    contact_frame: int,\n    window: int = 10,\n) -> float:\n    \"\"\"Gaussian proximity-weighted mean visibility of 8 key joints near contact.\"\"\"\n    KEY = [\"l_wrist\", \"r_wrist\", \"l_elbow\", \"r_elbow\",\n           \"l_shoulder\", \"r_shoulder\", \"l_hip\", \"r_hip\"]\n    sigma = max(window / 2.0, 1.0)\n    wv = tw = 0.0\n    for fi, lms in frame_results:\n        if lms is None or abs(fi - contact_frame) > window:\n            continue\n        fw = math.exp(-0.5 * ((fi - contact_frame) / sigma) ** 2)\n        for k in KEY:\n            wv += fw * lms[LM[k]].visibility\n            tw += fw\n    return round(wv / tw, 3) if tw > 1e-9 else 0.0\n\n\n# ---------------------------------------------------------------------------\n# Sidecar loading\n# ---------------------------------------------------------------------------\n\ndef load_sidecar(video_path: Path) -> Optional[dict]:\n    \"\"\"\n    Load the .yolo.json sidecar written by yolo_processing.py.\n    Returns None (with a warning) if the file does not exist.\n    \"\"\"\n    sidecar = video_path.with_suffix(\".yolo.json\")\n    if not sidecar.exists():\n        log.warning(\n            \"No sidecar found for %s \u2014 ball tracking disabled for this video.\\n\"\n            \"  Run yolo_processing.py first to generate %s\",\n            video_path.name, sidecar.name,\n        )\n        return None\n    with open(sidecar) as f:\n        return json.load(f)\n\n\ndef _sidecar_contact_frame(sidecar: Optional[dict]) -> Optional[int]:\n    \"\"\"Extract the YOLO-estimated contact frame from the sidecar.\"\"\"\n    if sidecar and \"contact_frame\" in sidecar:\n        return int(sidecar[\"contact_frame\"])\n    return None\n\n\ndef _sidecar_ball_fields(\n    sidecar: Optional[dict],\n) -> tuple[\n    Optional[float],\n    list[tuple[float, float]],\n    list[float],\n    list[tuple[float, float]],\n    list[float],\n    Optional[list],\n    Optional[str],\n]:\n    \"\"\"\n    Unpack ball-tracking fields from the YOLO sidecar into the format\n    expected by SwingResult.\n\n    Returns\n    -------\n    (ball_angle, post_centroids, post_radii, pre_centroids, pre_radii,\n     ball_contact_px, pitcher_side)\n    \"\"\"\n    if not sidecar:\n        return None, [], [], [], [], None, None\n\n    ball_angle     = sidecar.get(\"ball_angle\")\n    ball_contact   = sidecar.get(\"ball_contact_px\")      # [cx, cy] or None\n    pitcher_side   = sidecar.get(\"pitcher_side\")          # \"L\", \"R\", or None\n\n    post = sidecar.get(\"post_contact\", [])\n    pre  = sidecar.get(\"pre_contact\",  [])\n\n    post_centroids = [(d[\"cx\"], d[\"cy\"]) for d in post]\n    post_radii     = [d.get(\"r\", 8.0)   for d in post]\n    pre_centroids  = [(d[\"cx\"], d[\"cy\"]) for d in pre]\n    pre_radii      = [d.get(\"r\", 8.0)   for d in pre]\n\n    return ball_angle, post_centroids, post_radii, pre_centroids, pre_radii, ball_contact, pitcher_side\n\n\n# ---------------------------------------------------------------------------\n# Core pipeline\n# ---------------------------------------------------------------------------\n\ndef analyze_swing(\n    video_path: Path,\n    landmarker,\n    sidecar: Optional[dict] = None,\n    stride: int = STRIDE,\n    verbose: bool = VERBOSE,\n) -> tuple[SwingResult, list]:\n    \"\"\"\n    Full pose-based pipeline for one video.\n\n    1. Run MediaPipe pose estimation on every Nth frame.\n    2. Detect contact frame from wrist deceleration.\n       (Uses MediaPipe result; YOLO optical-flow estimate in sidecar is a cross-check.)\n    3. Compute launch angle from bat vector (pose) and blend with ball angle (YOLO sidecar).\n    4. Classify contact zone and landing zone.\n    5. Return (SwingResult, frame_results) \u2014 frame_results is reused for annotation.\n    \"\"\"\n    if verbose:\n        log.info(\"Processing: %s\", video_path.name)\n\n    _, meta = open_video(video_path)\n    fps = meta[\"fps\"]\n\n    # Stage 2: Pose estimation\n    # Anchor the pose window on the YOLO contact estimate so we only process\n    # the ±YOLO_ANCHOR_WINDOW frames most likely to contain the swing.\n    # Falls back to the 30%-85% clip gate when YOLO data is unavailable.\n    _, meta_fast = open_video(video_path)\n    total_frames = meta_fast[\"total_frames\"]\n    fps_fast     = meta_fast[\"fps\"]\n    yolo_cf_est  = _sidecar_contact_frame(sidecar)\n    if yolo_cf_est is not None:\n        pose_start = max(0,              yolo_cf_est - YOLO_ANCHOR_WINDOW)\n        pose_end   = min(total_frames-1, yolo_cf_est + YOLO_ANCHOR_WINDOW)\n        if verbose:\n            log.info(\"  Pose window anchored to YOLO cf=%d → frames %d-%d (of %d)\",\n                     yolo_cf_est, pose_start, pose_end, total_frames)\n    else:\n        pose_start = int(total_frames * 0.30)\n        pose_end   = int(total_frames * 0.85)\n        if verbose:\n            log.info(\"  No YOLO anchor — using 30%%–85%% gate: frames %d-%d\",\n                     pose_start, pose_end)\n\n    if verbose:\n        log.info(\"  Running pose estimation (stride=%d, window=%d frames)…\",\n                 stride, pose_end - pose_start + 1)\n    frame_results = run_pose_on_frames(video_path, landmarker, stride=stride,\n                                       frame_range=(pose_start, pose_end))\n\n    detected = sum(1 for _, lms in frame_results if lms is not None)\n    if verbose:\n        log.info(\"  Pose detected in %d / %d sampled frames\", detected, len(frame_results))\n    if detected == 0:\n        raise RuntimeError(\"No pose detected.\")\n\n    # Stage 3A: Contact frame (wrist deceleration \u2014 authoritative)\n    velocities    = compute_wrist_velocity(frame_results, fps / stride)\n    contact_frame = find_contact_frame(velocities)\n    contact_time  = contact_frame / fps\n\n    # Cross-check with YOLO estimate\n    yolo_cf = _sidecar_contact_frame(sidecar)\n    if yolo_cf is not None and abs(yolo_cf - contact_frame) > 20:\n        log.warning(\n            \"  Contact frame mismatch: pose=%d, YOLO=%d (diff=%d frames) \u2014 using pose\",\n            contact_frame, yolo_cf, abs(yolo_cf - contact_frame),\n        )\n\n    if verbose:\n        log.info(\"  Contact frame: %d (t=%.3fs)\", contact_frame, contact_time)\n\n    # Get landmarks at or near the contact frame\n    contact_lms = None\n    for fi, lms in frame_results:\n        if fi == contact_frame and lms is not None:\n            contact_lms = lms\n            break\n    if contact_lms is None:\n        best = min(\n            [(abs(fi - contact_frame), lms)\n             for fi, lms in frame_results if lms is not None],\n            key=lambda x: x[0],\n            default=(None, None),\n        )\n        contact_lms = best[1]\n    if contact_lms is None:\n        raise RuntimeError(\"No landmarks near contact frame.\")\n\n    # Stage 4: Launch angle geometry\n    handedness  = infer_handedness(contact_lms)\n    bat_vec     = compute_bat_vector(contact_lms, handedness)\n    sag_normal  = compute_sagittal_normal(contact_lms)\n    bat_proj    = project_sagittal(bat_vec, sag_normal)\n    world_angle = launch_angle_world(bat_proj)\n    image_angle = launch_angle_image(contact_lms, handedness)\n\n    # 70% image-space (reliable X/Y) + 30% world-space\n    pose_angle = (\n        round(0.70 * image_angle + 0.30 * world_angle, 2)\n        if image_angle is not None else world_angle\n    )\n\n    if verbose:\n        log.info(\n            \"  Handedness: %s | image: %s | world: %.1f\u00b0 | pose blend: %.1f\u00b0\",\n            handedness,\n            f\"{image_angle:.1f}\u00b0\" if image_angle is not None else \"n/a\",\n            world_angle,\n            pose_angle,\n        )\n\n    # Stage 3B: Ball angle and tracking data from YOLO sidecar\n    (ball_angle, ball_centroids, ball_radii,\n     pre_centroids, pre_radii, ball_contact_px,\n     pitcher_side) = _sidecar_ball_fields(\n        sidecar if ENABLE_BALL_BLEND else None\n    )\n\n    if ball_angle is not None and verbose:\n        log.info(\"  Ball angle (YOLO sidecar): %.1f\u00b0  [pitcher_side=%s]\",\n                 ball_angle, pitcher_side or \"unset\")\n\n    # \u2500\u2500 Pitcher-side angle sign validation \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    # A correctly tracked post-contact ball must produce a launch angle whose\n    # horizontal component travels *away* from the pitcher's release side.\n    # If the sign is flipped (e.g. YOLO latched onto a pre-contact frame or\n    # a background blob moving the wrong way) we null the ball_angle so the\n    # blended result is not corrupted by a mirrored measurement.\n    if ball_angle is not None and pitcher_side is not None:\n        # The angle is from atan2(vy, |vx|) so it lives in [-90, +90].\n        # The horizontal sign was already normalised in _fit_launch_angle,\n        # but the extreme angles (close to \u00b190\u00b0) suggest the horizontal\n        # component is near-zero \u2014 those are still valid.\n        # Cross-check pose_angle sign: if they differ by more than 70\u00b0 the\n        # ball angle is likely bad.\n        sign_diff = abs(ball_angle - pose_angle)\n        if sign_diff > 70.0:\n            log.warning(\n                \"  Ball angle %.1f\u00b0 vs pose angle %.1f\u00b0 \u2014 sign diff=%.0f\u00b0 > 70\u00b0 \"\n                \"(pitcher_side=%s): ball_angle nulled to prevent bad blend\",\n                ball_angle, pose_angle, sign_diff, pitcher_side,\n            )\n            ball_angle = None\n\n    # Blend: pose 60%, ball 40% when ball data available\n    blended = (\n        round(0.60 * pose_angle + 0.40 * ball_angle, 2)\n        if ball_angle is not None else pose_angle\n    )\n\n    # Stage 5: Classification\n    landing_zone, outcome_desc = angle_to_landing_zone(blended)\n    cp_zone     = contact_point_zone(contact_lms, handedness)\n    confidence  = estimate_confidence(frame_results, contact_frame)\n\n    result = SwingResult(\n        video_path=str(video_path),\n        contact_frame=contact_frame,\n        contact_time_s=round(contact_time, 3),\n        launch_angle=blended,\n        bat_vector=bat_vec.tolist(),\n        contact_zone=cp_zone,\n        landing_zone=landing_zone,\n        outcome_description=outcome_desc,\n        confidence=confidence,\n        batter_handedness=handedness,\n        ball_angle=ball_angle,\n        blended_angle=blended,\n        ball_centroids=ball_centroids,\n        ball_radii=ball_radii,\n        pre_contact_centroids=pre_centroids,\n        pre_contact_radii=pre_radii,\n        ball_contact_px=ball_contact_px,\n        pitcher_side=pitcher_side,\n    )\n    return result, frame_results\n\n\n# ---------------------------------------------------------------------------\n# Annotation helpers\n# ---------------------------------------------------------------------------\n\ndef _validate_ball_contact_px(\n    ball_contact_px: Optional[list],\n    lms: Optional[list[Landmark3D]],\n    w: int,\n    h: int,\n    max_dist_px: float = MAX_CONTACT_WRIST_DIST_PX,\n) -> bool:\n    \"\"\"\n    Return True only if ball_contact_px is plausibly close to the wrist\n    midpoint in pose landmarks.\n\n    Why this matters: the YOLO sidecar stores the *first post-contact ball\n    detection* as ball_contact_px. If the tracker latched onto a background\n    object (crowd movement, helmet, jersey number) instead of the actual ball,\n    that pixel can be anywhere in the frame \u2014 often hundreds of pixels from\n    where the bat actually met the ball. Using it unchecked places the CONTACT\n    annotation ring on background motion rather than the ball/bat collision.\n\n    We cross-check against the wrist midpoint from MediaPipe pose, which is\n    far more reliable for pinpointing the contact region. If the two sources\n    disagree by more than max_dist_px we treat ball_contact_px as unreliable.\n    \"\"\"\n    if ball_contact_px is None or lms is None:\n        return False\n    OFF = 33 if len(lms) >= 66 else 0\n    needed = OFF + max(LM[\"l_wrist\"], LM[\"r_wrist\"]) + 1\n    if len(lms) < needed:\n        return False\n    lw = lms[OFF + LM[\"l_wrist\"]]\n    rw = lms[OFF + LM[\"r_wrist\"]]\n    wx = ((lw.x + rw.x) / 2) * w\n    wy = ((lw.y + rw.y) / 2) * h\n    bx = float(ball_contact_px[0])\n    by = float(ball_contact_px[1])\n    dist = math.sqrt((bx - wx) ** 2 + (by - wy) ** 2)\n    return dist <= max_dist_px\n\n\ndef _filter_trail_outliers(\n    trail: list[tuple[int, int, float, bool]],\n    max_residual_px: float = 50.0,\n) -> list[tuple[int, int, float, bool]]:\n    \"\"\"\n    Remove trail points that deviate more than max_residual_px from a\n    parabolic fit to the trail's y-coordinates.\n\n    A real ball trajectory is well-described by y(t) = a\u00b7t\u00b2 + b\u00b7t + c.\n    Points that fall far outside this curve are background blobs that\n    the tracker momentarily mistook for the ball. Removing them before\n    rendering prevents the trail from visually snapping to background\n    objects and immediately returning to the ball path.\n\n    Falls back to the unfiltered trail when fewer than 5 points are\n    available (not enough to fit a parabola reliably).\n    \"\"\"\n    if len(trail) < 5:\n        return trail\n    xs = np.array([p[0] for p in trail], dtype=float)\n    ys = np.array([p[1] for p in trail], dtype=float)\n    ts = np.arange(len(trail), dtype=float)\n    try:\n        poly      = np.polyfit(ts, ys, 2)\n        y_pred    = np.polyval(poly, ts)\n        residuals = np.abs(ys - y_pred)\n        filtered  = [p for p, r in zip(trail, residuals) if r <= max_residual_px]\n        # Keep at least 3 points so the trail doesn't vanish entirely\n        return filtered if len(filtered) >= 3 else trail\n    except Exception:\n        return trail\n\n\n# ---------------------------------------------------------------------------\n# Stage 6: Annotation helpers (continued)\n# ---------------------------------------------------------------------------\n\ndef _contact_pixel(\n    lms: list[Landmark3D],\n    handedness: str,\n    w: int,\n    h: int,\n    ball_contact_px: Optional[list] = None,\n) -> tuple[int, int]:\n    \"\"\"\n    Return the pixel coordinate for the contact annotation.\n\n    Priority:\n    1. ball_contact_px from YOLO sidecar \u2014 ONLY when it passes a proximity\n       check against the wrist midpoint. If it's far from the wrist it almost\n       certainly came from a background false positive (the tracker latched onto\n       crowd motion), which is what caused the CONTACT ring to appear on\n       background movement rather than on the bat/ball collision point.\n    2. Wrist midpoint from pose \u2014 reliable fallback even when YOLO tracking\n       was poor.\n    \"\"\"\n    if ball_contact_px is not None and _validate_ball_contact_px(\n        ball_contact_px, lms, w, h\n    ):\n        return int(ball_contact_px[0]), int(ball_contact_px[1])\n    OFFSET = 33 if len(lms) >= 66 else 0\n    lw = lms[OFFSET + LM[\"l_wrist\"]]\n    rw = lms[OFFSET + LM[\"r_wrist\"]]\n    return int(((lw.x + rw.x) / 2) * w), int(((lw.y + rw.y) / 2) * h)\n\n\ndef draw_frame(\n    frame: np.ndarray,\n    lms: Optional[list[Landmark3D]],\n    result: SwingResult,\n    frame_idx: int,\n    post_contact: bool,\n    h: int,\n    w: int,\n    ball_trail: list[tuple[int, int, float, bool]],   # (cx, cy, r, is_predicted)\n    pre_ball: Optional[tuple[int, int, float]],\n    contact_px: Optional[tuple[int, int]],\n    bat_bbox: Optional[list[float]],\n    sidecar: Optional[dict],\n) -> np.ndarray:\n    \"\"\"Render all annotation layers onto a single frame.\"\"\"\n    out = frame.copy()\n\n    # 1. Skeleton\n    if lms is not None and len(lms) >= 66:\n        OFF  = 33\n        CONN = [\n            (\"l_shoulder\", \"r_shoulder\"), (\"l_shoulder\", \"l_elbow\"),\n            (\"l_elbow\",    \"l_wrist\"),    (\"r_shoulder\", \"r_elbow\"),\n            (\"r_elbow\",    \"r_wrist\"),    (\"l_shoulder\", \"l_hip\"),\n            (\"r_shoulder\", \"r_hip\"),      (\"l_hip\",      \"r_hip\"),\n            (\"l_hip\",      \"l_knee\"),     (\"r_hip\",      \"r_knee\"),\n            (\"l_knee\",     \"l_ankle\"),    (\"r_knee\",     \"r_ankle\"),\n        ]\n        col = _COL_SKEL_POST if post_contact else _COL_SKEL_PRE\n        for ak, bk in CONN:\n            a = lms[OFF + LM[ak]]; b = lms[OFF + LM[bk]]\n            ax, ay = int(a.x * w), int(a.y * h)\n            bx, by = int(b.x * w), int(b.y * h)\n            if all(0 <= v < lim for v, lim in [(ax, w), (bx, w), (ay, h), (by, h)]):\n                cv2.line(out, (ax, ay), (bx, by), col, 2, cv2.LINE_AA)\n        for k in LM:\n            lm = lms[OFF + LM[k]]\n            px, py = int(lm.x * w), int(lm.y * h)\n            if 0 <= px < w and 0 <= py < h:\n                cv2.circle(out, (px, py), 4, col, -1, cv2.LINE_AA)\n\n    # 2. Bat bounding box (from YOLO sidecar)\n    if bat_bbox is not None:\n        x1, y1, x2, y2 = (int(v) for v in bat_bbox)\n        cv2.rectangle(out, (x1, y1), (x2, y2), _COL_BAT_BOX, 2, cv2.LINE_AA)\n        cv2.putText(out, \"BAT\", (x1, max(0, y1 - 6)),\n                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, _COL_BAT_BOX, 1, cv2.LINE_AA)\n\n    # 3. Incoming pitch circle (pre-contact)\n    if pre_ball is not None and not post_contact:\n        bx, by, br = pre_ball\n        r = max(6, int(br))\n        ov = out.copy()\n        cv2.circle(ov, (bx, by), r, (255, 255, 255), -1, cv2.LINE_AA)\n        cv2.addWeighted(ov, 0.25, out, 0.75, 0, out)\n        cv2.circle(out, (bx, by), r, (255, 255, 255), 2, cv2.LINE_AA)\n        cv2.putText(out, \"PITCH\", (bx + r + 4, by - r),\n                    cv2.FONT_HERSHEY_SIMPLEX, 0.48, (200, 200, 200), 1, cv2.LINE_AA)\n\n    # 4. Contact marker \u2014 frozen pulsing rings at the original contact pixel,\n    #    plus a live wrist-tracking dot that follows the batter through follow-through.\n    #    The frozen ring fades after 25 frames; the live dot persists until clip end.\n    if contact_px is not None and post_contact:\n        cx, cy = contact_px\n        age    = frame_idx - result.contact_frame\n\n        # Frozen pulsing rings at the exact contact location (fade over 25 frames)\n        for rr in [10, 20, 32]:\n            alpha = max(0.0, 1.0 - age / 25.0)\n            if alpha <= 0:\n                break\n            ov = out.copy()\n            cv2.circle(ov, (cx, cy), rr, _COL_CONTACT, 2, cv2.LINE_AA)\n            cv2.addWeighted(ov, alpha, out, 1 - alpha, 0, out)\n        cv2.circle(out, (cx, cy), 5, _COL_CONTACT, -1, cv2.LINE_AA)\n        if age <= 25:\n            cv2.putText(out, \"CONTACT\", (cx + 12, cy - 12),\n                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, _COL_CONTACT, 1, cv2.LINE_AA)\n\n    # Live wrist-tracking dot: follows batter through the follow-through using\n    # per-frame landmarks so the annotation stays visually attached to the batter.\n    if lms is not None and len(lms) >= 66 and post_contact:\n        OFF = 33\n        lw_lm = lms[OFF + LM[\"l_wrist\"]]\n        rw_lm = lms[OFF + LM[\"r_wrist\"]]\n        wx = int(((lw_lm.x + rw_lm.x) / 2) * w)\n        wy = int(((lw_lm.y + rw_lm.y) / 2) * h)\n        if 0 <= wx < w and 0 <= wy < h:\n            cv2.circle(out, (wx, wy), 6, _COL_CONTACT, 2, cv2.LINE_AA)\n\n    # 5. Post-contact ball trail + live ball circle with homing annotation\n    # Filter outliers (background blobs that briefly hijacked the tracker)\n    # before rendering \u2014 prevents the trail from visually snapping to crowd\n    # objects and back.\n    rendered_trail = _filter_trail_outliers(ball_trail) if len(ball_trail) >= 5 else ball_trail\n    if len(rendered_trail) >= 2:\n        n = len(rendered_trail)\n        for i in range(1, n):\n            t      = i / n\n            _, _, _, is_pred = rendered_trail[i]\n            # Predicted (Kalman-only) segments render dimmer and dashed\n            if is_pred:\n                trail_col = (0, int(80 * t), int(130 * t))\n                thickness = 1\n            else:\n                trail_col = (0, int(180 * t), int(255 * t))\n                thickness = max(1, int(3 * t))\n            cv2.line(out,\n                     (rendered_trail[i-1][0], rendered_trail[i-1][1]),\n                     (rendered_trail[i][0],   rendered_trail[i][1]),\n                     trail_col, thickness, cv2.LINE_AA)\n\n    if rendered_trail:\n        bx, by, br, is_last_pred = rendered_trail[-1]\n        r  = max(6, int(br))\n\n        if is_last_pred:\n            # \u2500\u2500 Kalman-predicted position: dashed circle + crosshair \u2500\u2500\u2500\u2500\u2500\u2500\n            # Draw dashed circle by rendering arc segments\n            for seg in range(0, 360, 30):\n                a1 = math.radians(seg)\n                a2 = math.radians(seg + 20)\n                p1 = (int(bx + r * math.cos(a1)), int(by + r * math.sin(a1)))\n                p2 = (int(bx + r * math.cos(a2)), int(by + r * math.sin(a2)))\n                cv2.line(out, p1, p2, (0, 160, 200), 1, cv2.LINE_AA)\n            # Crosshair \u2014 smaller, dimmer\n            arm = r + 8\n            cv2.line(out, (bx - arm, by), (bx + arm, by), (0, 160, 200), 1, cv2.LINE_AA)\n            cv2.line(out, (bx, by - arm), (bx, by + arm), (0, 160, 200), 1, cv2.LINE_AA)\n            cv2.putText(out, \"~PRED\", (bx + r + 4, by - r),\n                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 160, 200), 1, cv2.LINE_AA)\n        else:\n            # \u2500\u2500 Confirmed detection: solid circle + homing crosshair \u2500\u2500\u2500\u2500\u2500\u2500\n            ov = out.copy()\n            cv2.circle(ov, (bx, by), r, _COL_BALL, -1, cv2.LINE_AA)\n            cv2.addWeighted(ov, 0.35, out, 0.65, 0, out)\n            cv2.circle(out, (bx, by), r, _COL_BALL, 2, cv2.LINE_AA)\n            cv2.circle(out, (bx, by), r, (255, 255, 255), 1, cv2.LINE_AA)\n\n            # Homing crosshair with gap (doesn't overlap the circle)\n            gap  = r + 4\n            arm  = r + 20\n            col  = (0, 255, 220)\n            # Horizontal arms\n            cv2.line(out, (bx - arm, by), (bx - gap, by), col, 1, cv2.LINE_AA)\n            cv2.line(out, (bx + gap, by), (bx + arm, by), col, 1, cv2.LINE_AA)\n            # Vertical arms\n            cv2.line(out, (bx, by - arm), (bx, by - gap), col, 1, cv2.LINE_AA)\n            cv2.line(out, (bx, by + gap), (bx, by + arm), col, 1, cv2.LINE_AA)\n            # Corner ticks on the circle for a targeting-reticle look\n            for ang_deg in [45, 135, 225, 315]:\n                ang = math.radians(ang_deg)\n                ix  = int(bx + r * math.cos(ang))\n                iy  = int(by + r * math.sin(ang))\n                ox  = int(bx + (r + 8) * math.cos(ang))\n                oy  = int(by + (r + 8) * math.sin(ang))\n                cv2.line(out, (ix, iy), (ox, oy), col, 1, cv2.LINE_AA)\n\n    # 6. Launch angle ray \u2014 originates from live wrist midpoint so it stays\n    #    visually anchored to the batter as they move through follow-through.\n    #    Falls back to frozen contact_px if landmarks aren't available.\n    _ray_origin: Optional[tuple[int, int]] = None\n    if lms is not None and len(lms) >= 66 and post_contact:\n        OFF = 33\n        lw_lm = lms[OFF + LM[\"l_wrist\"]]\n        rw_lm = lms[OFF + LM[\"r_wrist\"]]\n        rwx = int(((lw_lm.x + rw_lm.x) / 2) * w)\n        rwy = int(((lw_lm.y + rw_lm.y) / 2) * h)\n        if 0 <= rwx < w and 0 <= rwy < h:\n            _ray_origin = (rwx, rwy)\n    if _ray_origin is None and contact_px is not None:\n        _ray_origin = contact_px\n\n    if _ray_origin is not None and post_contact and result.ball_angle is not None:\n        cx, cy  = _ray_origin\n        angle_r = math.radians(result.ball_angle)\n\n        # Determine the horizontal direction the ball travels after contact.\n        # R-pitcher \u2192 ball was coming from the left \u2192 post-contact travels right (+x)\n        # L-pitcher \u2192 ball was coming from the right \u2192 post-contact travels left  (-x)\n        # None      \u2192 default to right (+x) for the ray (safe visual fallback)\n        pitcher_side_r = getattr(result, \"pitcher_side\", None)\n        if pitcher_side_r is not None:\n            ray_x_sign = +1 if pitcher_side_r.upper() == \"R\" else -1\n        else:\n            ray_x_sign = +1\n\n        ray_len = 200\n        ex = int(cx + ray_x_sign * ray_len * math.cos(angle_r))\n        ey = int(cy - ray_len * math.sin(angle_r))   # image y-up correction\n\n        # Draw the primary launch-angle ray\n        cv2.arrowedLine(out, (cx, cy), (ex, ey), _COL_RAY, 2, cv2.LINE_AA, tipLength=0.12)\n\n        # Angle label \u2014 include pitcher_side tag so the reader knows which\n        # direction convention was used when interpreting the angle.\n        side_tag = f\"[P:{pitcher_side_r}]\" if pitcher_side_r else \"\"\n        label    = f\"{result.ball_angle:+.1f}\u00b0 {side_tag}\"\n        # Position label so it doesn't overlap the arrowhead\n        lx = ex + (8 * ray_x_sign)\n        ly = ey\n        cv2.putText(out, label, (lx, ly),\n                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, _COL_RAY, 1, cv2.LINE_AA)\n\n        # Secondary ghost ray at the pose-based angle for cross-check\n        # (drawn dimmer \u2014 helps spot when ball and pose angles diverge)\n        pose_angle_val = result.launch_angle   # blended; close to pose when ball=None\n        if result.ball_angle is not None and abs(result.ball_angle - pose_angle_val) > 3.0:\n            pa_r = math.radians(pose_angle_val)\n            pe_x = int(cx + ray_x_sign * 140 * math.cos(pa_r))\n            pe_y = int(cy - 140 * math.sin(pa_r))\n            cv2.arrowedLine(out, (cx, cy), (pe_x, pe_y),\n                            (120, 60, 255), 1, cv2.LINE_AA, tipLength=0.10)\n            cv2.putText(out, f\"pose {pose_angle_val:+.1f}\u00b0\",\n                        (pe_x + 4 * ray_x_sign, pe_y - 6),\n                        cv2.FONT_HERSHEY_SIMPLEX, 0.42, (120, 60, 255), 1, cv2.LINE_AA)\n\n    # 7. HUD panel\n    if post_contact:\n        ov = out.copy()\n        cv2.rectangle(ov, (12, 12), (362, 177), _COL_HUD_BG, -1)\n        cv2.addWeighted(ov, 0.65, out, 0.35, 0, out)\n        cv2.rectangle(out, (12, 12), (362, 177), (80, 80, 80), 1)\n        cp_src       = \"ball\" if result.ball_contact_px else \"pose\"\n        angle_src    = \"\" if result.ball_angle is not None else \" [pose only]\"\n        contact_time = f\"{result.contact_time_s:.2f}s\"\n        lines  = [\n            (f\"** CONTACT  t={contact_time}  [{cp_src}]\",            (0, 230, 200)),\n            (f\"  Launch angle : {result.launch_angle:+.1f}deg{angle_src}\", (220, 220, 220)),\n            (f\"  Ball angle   : {result.ball_angle:+.1f}deg\"\n             if result.ball_angle else \"  Ball angle   : n/a\",        (220, 220, 220)),\n            (f\"  Landing zone : {result.outcome_description}\",        (220, 220, 220)),\n            (f\"  Contact zone : {result.contact_zone}\",               (220, 220, 220)),\n            (f\"  Hand: {result.batter_handedness}   Conf: {result.confidence:.0%}\",\n                                                                       (160, 160, 160)),\n        ]\n        for row, (txt, col) in enumerate(lines):\n            cv2.putText(out, txt, (22, 42 + row * 22),\n                        cv2.FONT_HERSHEY_SIMPLEX, 0.52, col, 1, cv2.LINE_AA)\n\n    # 8. Frame counter\n    cv2.putText(out, f\"frame {frame_idx}\", (w - 130, h - 14),\n                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (140, 140, 140), 1, cv2.LINE_AA)\n    return out\n\n\ndef write_annotated_video(\n    video_path: Path,\n    result: SwingResult,\n    frame_results: list,\n    out_path: Path,\n    sidecar: Optional[dict] = None,\n    contact_window: int = 90,\n) -> None:\n    \"\"\"\n    Write an annotated clip (\u00b1contact_window frames around contact).\n    Ball trail, bat boxes, and contact pixel come from the YOLO sidecar.\n    Skeleton overlay comes from MediaPipe frame_results.\n    \"\"\"\n    cap, meta = open_video(video_path)\n    fps, w, h = meta[\"fps\"], meta[\"width\"], meta[\"height\"]\n    writer    = cv2.VideoWriter(\n        str(out_path), cv2.VideoWriter_fourcc(*\"mp4v\"), fps, (w, h)\n    )\n\n    lms_map   = {fi: lms for fi, lms in frame_results}\n    start     = max(0, result.contact_frame - contact_window)\n    end       = result.contact_frame + contact_window\n\n    # Contact pixel: ball-derived > wrist-fallback\n    c_lms      = lms_map.get(result.contact_frame)\n    contact_px = None\n    if c_lms and len(c_lms) >= 66:\n        contact_px = _contact_pixel(\n            c_lms, result.batter_handedness, w, h, result.ball_contact_px\n        )\n\n    # Frame-indexed lookups from sidecar\n    # Each entry: (cx, cy, r, is_predicted)\n    # is_predicted=True means the position came from Kalman extrapolation (no detection)\n    post_map: dict[int, tuple[int, int, float, bool]] = {}\n    post_dets_raw = sidecar.get(\"post_contact\", []) if sidecar else []\n    radii = result.ball_radii or [8.0] * len(result.ball_centroids)\n    for i, ((cx, cy), r) in enumerate(zip(result.ball_centroids, radii)):\n        is_pred = False\n        if i < len(post_dets_raw):\n            is_pred = bool(post_dets_raw[i].get(\"predicted\", False))\n        frame_key = int(post_dets_raw[i][\"frame\"]) if i < len(post_dets_raw)                     else result.contact_frame + i\n        post_map[frame_key] = (int(cx), int(cy), float(r), is_pred)\n\n    pre_map: dict[int, tuple[int, int, float]] = {}\n    n_pre = len(result.pre_contact_centroids)\n    p_radii = result.pre_contact_radii or [8.0] * n_pre\n    for i, ((cx, cy), r) in enumerate(zip(result.pre_contact_centroids, p_radii)):\n        pre_map[result.contact_frame - n_pre + i] = (int(cx), int(cy), float(r))\n\n    bat_boxes: dict[int, list[float]] = {}\n    if sidecar and \"bat_boxes\" in sidecar:\n        bat_boxes = {int(k): v for k, v in sidecar[\"bat_boxes\"].items()}\n\n    cap.set(cv2.CAP_PROP_POS_FRAMES, start)\n    # trail entries: (cx, cy, r, is_predicted)\n    trail: list[tuple[int, int, float, bool]] = []\n    # Track the last TWO confirmed (non-predicted) positions to compute a\n    # running velocity for direction-consistency filtering.  This prevents\n    # the trail from jumping to a background object and then snapping back\n    # to the ball, which creates a visible zig-zag artefact.\n    _trail_prev1: Optional[tuple[int, int]] = None   # most-recent confirmed point\n    _trail_prev2: Optional[tuple[int, int]] = None   # second-most-recent confirmed\n\n    for fi in range(start, end + 1):\n        ret, frame = cap.read()\n        if not ret:\n            break\n\n        post_contact = fi >= result.contact_frame\n        if fi in post_map:\n            candidate = post_map[fi]\n            cx_c, cy_c, _, is_pred_c = candidate\n            accept = True\n\n            # Direction-consistency check \u2014 only for confirmed (non-predicted)\n            # detections once we have at least two prior confirmed positions.\n            if not is_pred_c and _trail_prev1 is not None and _trail_prev2 is not None:\n                # Current velocity direction (from prev2 \u2192 prev1)\n                vx_hist = _trail_prev1[0] - _trail_prev2[0]\n                vy_hist = _trail_prev1[1] - _trail_prev2[1]\n                v_hist_mag = math.sqrt(vx_hist ** 2 + vy_hist ** 2)\n                # Proposed step direction (from prev1 \u2192 candidate)\n                vx_new = cx_c - _trail_prev1[0]\n                vy_new = cy_c - _trail_prev1[1]\n                v_new_mag = math.sqrt(vx_new ** 2 + vy_new ** 2)\n                if v_hist_mag > 1.0 and v_new_mag > 1.0:\n                    cos_angle = (\n                        (vx_hist * vx_new + vy_hist * vy_new)\n                        / (v_hist_mag * v_new_mag)\n                    )\n                    # Reject if direction reverses by more than 120\u00b0\n                    if cos_angle < math.cos(math.radians(120)):\n                        accept = False\n\n            if accept:\n                trail.append(candidate)\n                if not is_pred_c:\n                    _trail_prev2 = _trail_prev1\n                    _trail_prev1 = (cx_c, cy_c)\n\n        annotated = draw_frame(\n            frame,\n            lms_map.get(fi),\n            result, fi,\n            post_contact, h, w,\n            ball_trail=list(trail),\n            pre_ball=pre_map.get(fi) if not post_contact else None,\n            contact_px=contact_px if post_contact else None,\n            bat_bbox=bat_boxes.get(fi),\n            sidecar=sidecar,\n        )\n        writer.write(annotated)\n\n    cap.release()\n    writer.release()\n    log.info(\"  Annotated video \u2192 %s\", out_path.name)\n\n\n# ---------------------------------------------------------------------------\n# Main\n# ---------------------------------------------------------------------------\n\ndef main() -> None:\n    # Handle --download-model flag\n    if \"--download-model\" in sys.argv:\n        download_model()\n        return\n\n    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)\n    MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)\n    download_model()\n\n    # Find all videos (skip annotated clips and outputs folder)\n    all_videos = sorted(\n        p for p in VIDEO_ROOT.rglob(\"*\")\n        if p.is_file()\n        and p.suffix.lower() in VIDEO_EXTENSIONS\n        and \"_annotated\" not in p.stem\n        and OUTPUT_DIR not in p.parents\n    )\n\n    if not all_videos:\n        log.error(\"No videos found in %s\", VIDEO_ROOT)\n        sys.exit(1)\n\n    if PROOF_CHECK:\n        videos = all_videos[:PROOF_CHECK_N]\n        log.info(\n            \"[proof-check] Processing first %d / %d videos with annotation\",\n            len(videos), len(all_videos),\n        )\n    else:\n        videos = all_videos\n        log.info(\"Found %d video(s)\", len(videos))\n\n    results: list[SwingResult] = []\n    failed:  list[tuple[str, str]] = []\n\n    for i, vp in enumerate(videos, 1):\n        log.info(\"[%d/%d] %s\", i, len(videos), vp.name)\n\n        # Load YOLO sidecar (may be None if yolo_processing.py hasn't run yet)\n        sidecar = load_sidecar(vp)\n\n        # Fresh landmarker per video \u2014 monotonically increasing timestamp requirement\n        landmarker = load_pose_landmarker()\n        try:\n            result, frame_results = analyze_swing(\n                vp, landmarker, sidecar=sidecar, verbose=VERBOSE\n            )\n\n            if PROOF_CHECK:\n                ann_out = vp.parent / (vp.stem + \"_annotated.mp4\")\n                write_annotated_video(vp, result, frame_results, ann_out, sidecar)\n\n        except Exception as e:\n            log.warning(\"  FAILED: %s\", e)\n            failed.append((vp.name, str(e)))\n            continue\n        finally:\n            landmarker.close()\n\n        results.append(result)\n\n        # Per-video JSON\n        json_out = OUTPUT_DIR / (vp.stem + \"_result.json\")\n        with open(json_out, \"w\") as f:\n            json.dump(asdict(result), f, indent=2)\n        log.info(\"  \u2192 JSON: %s\", json_out.name)\n\n    # Summary CSV\n    if results:\n        BATCH_CSV.parent.mkdir(parents=True, exist_ok=True)\n        fieldnames = list(asdict(results[0]).keys())\n        with open(BATCH_CSV, \"w\", newline=\"\") as f:\n            writer = csv.DictWriter(f, fieldnames=fieldnames)\n            writer.writeheader()\n            for r in results:\n                row = asdict(r)\n                for list_field in (\"bat_vector\", \"ball_centroids\",\n                                   \"pre_contact_centroids\"):\n                    row[list_field] = json.dumps(row[list_field])\n                writer.writerow(row)\n        log.info(\"Summary CSV \u2192 %s\", BATCH_CSV)\n\n    mode = \"proof-check\" if PROOF_CHECK else \"batch\"\n    print(f\"\\n\u2714 [{mode}] {len(results)}/{len(videos)} videos processed \u2192 {OUTPUT_DIR}\")\n    if PROOF_CHECK:\n        print(f\"  Annotated clips in: {VIDEO_ROOT}\")\n    if failed:\n        print(f\"\u2717 Failed ({len(failed)}):\")\n        for name, err in failed:\n            print(f\"    {name}: {err}\")\n\n\nif __name__ == \"__main__\":\n    main()"
with open('/content/mediapipe_processing.py','w') as f: f.write(mp_src)
print('mediapipe_processing.py written')

mediapipe_processing.py written


## 4. Download MediaPipe PoseLandmarker model (Lite)

In [ ]:
import urllib.request
# Using pose_landmarker_LITE: ~3x faster than heavy, sufficient for wrist localisation
MODEL_PATH = MODEL_DIR / 'pose_landmarker_lite.task'
MODEL_URL  = (
    'https://storage.googleapis.com/mediapipe-models/'
    'pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task'
)

# Ensure the target directory exists before downloading
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

if not MODEL_PATH.exists() or MODEL_PATH.stat().st_size < 1_000_000:
    print('Downloading PoseLandmarker (~25 MB)...')
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
print(f'Model ready: {MODEL_PATH.stat().st_size/1e6:.1f} MB')

Model ready: 5.8 MB


## 2b. Trim videos to swing window using hip + arm motion

**Problem with the old trim:** The previous approach trimmed ±10s around a motion-energy
peak, which often captured 3+ seconds of the batter standing still before any swing motion.
This polluted the training patches with frames where the contact zone is empty.

**New approach — hip & arm motion gate:**
For each video we scan forward frame-by-frame and find the first frame where *both*:
- The **hip midpoint** has moved ≥ `HIP_MOTION_PX` pixels from its initial position, AND
- The **wrist midpoint** has moved ≥ `WRIST_MOTION_PX` pixels from its initial position

This is the anatomical onset of the swing load — the batter shifts weight and raises
hands simultaneously. We then keep `PRE_SWING_S` seconds before that point and
`POST_SWING_S` seconds after it, writing a trimmed clip back to Drive.

Already-trimmed clips (named `_swing.mp4`) are skipped.

In [ ]:
import cv2, sys, shutil, subprocess
import numpy as np
from pathlib import Path
from tqdm import tqdm

# ── Trim parameters ──────────────────────────────────────────────────────────
HIP_MOTION_PX   = 12    # hip midpoint must move this many pixels to count as swing start
WRIST_MOTION_PX = 20    # wrist midpoint must move this many pixels to count as swing start
PRE_SWING_S     = 1.5   # seconds to keep before the detected swing onset
POST_SWING_S    = 4.5   # seconds to keep after swing onset (covers full swing + follow-through)
STRIDE_SCAN     = 3     # check every Nth frame when scanning for motion (speed-up)
MIN_CLIP_S      = 4.0   # reject trim if result would be shorter than this
TRIM_SUFFIX     = '_swing'  # output filename suffix

# MediaPipe lite model for hip/wrist detection during scanning
sys.path.insert(0, '/content')
import mediapipe_processing as mp_mod
mp_mod.MODEL_PATH = MODEL_PATH

def _find_swing_onset(video_path: Path, fps: float, total: int) -> int | None:
    """
    Scan the video and return the frame index where both hip and wrist
    motion exceed their thresholds simultaneously.

    Uses MediaPipe landmarks (hip + wrist midpoints) so the gate is based
    on actual body-part motion, not pixel-level scene changes.

    Returns None if no clear onset is found (video will be left untrimmed).
    """
    landmarker = mp_mod.load_pose_landmarker(mp_mod.MODEL_PATH)
    cap        = cv2.VideoCapture(str(video_path))
    LM         = mp_mod.LM

    ref_hip   = None   # (x, y) reference hip midpoint in image-normalised coords
    ref_wrist = None   # (x, y) reference wrist midpoint

    onset_frame = None

    try:
        for fi in range(0, total, STRIDE_SCAN):
            cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
            ret, frame = cap.read()
            if not ret:
                break

            h, w = frame.shape[:2]
            padded, pad_top, pad_left = mp_mod._pad_to_square(frame)
            rgb = cv2.cvtColor(padded, cv2.COLOR_BGR2RGB)

            import mediapipe as mp
            mp_img    = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            ts_ms     = int(fi / fps * 1000)
            pose_res  = landmarker.detect_for_video(mp_img, ts_ms)

            if not pose_res.pose_landmarks:
                continue

            # Use image-space landmarks (un-padded) for pixel-level motion check
            raw_lms = pose_res.pose_landmarks[0]
            size    = max(h, w)
            def px(lm_idx):
                lm = raw_lms[lm_idx]
                px_x = lm.x * size - pad_left
                px_y = lm.y * size - pad_top
                return px_x / w * w, px_y / h * h  # back to original pixel coords

            lh_x, lh_y = px(LM['l_hip'])
            rh_x, rh_y = px(LM['r_hip'])
            lw_x, lw_y = px(LM['l_wrist'])
            rw_x, rw_y = px(LM['r_wrist'])

            hip_mid   = ((lh_x + rh_x) / 2, (lh_y + rh_y) / 2)
            wrist_mid = ((lw_x + rw_x) / 2, (lw_y + rw_y) / 2)

            # Set reference from the first detected frame
            if ref_hip is None:
                ref_hip   = hip_mid
                ref_wrist = wrist_mid
                continue

            hip_dist   = np.hypot(hip_mid[0]   - ref_hip[0],   hip_mid[1]   - ref_hip[1])
            wrist_dist = np.hypot(wrist_mid[0] - ref_wrist[0], wrist_mid[1] - ref_wrist[1])

            if hip_dist >= HIP_MOTION_PX and wrist_dist >= WRIST_MOTION_PX:
                onset_frame = fi
                break

    finally:
        cap.release()
        landmarker.close()

    return onset_frame


def trim_to_swing(video_path: Path) -> Path | None:
    """
    Trim one video around the detected swing onset.
    Returns the output path on success, None if skipped or failed.
    """
    out_path = video_path.parent / (video_path.stem + TRIM_SUFFIX + video_path.suffix)
    if out_path.exists():
        return out_path   # already trimmed

    cap   = cv2.VideoCapture(str(video_path))
    fps   = cap.get(cv2.CAP_PROP_FPS) or 60.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    dur_s = total / fps
    cap.release()

    onset = _find_swing_onset(video_path, fps, total)

    if onset is None:
        # Fallback: use 30-85% energy-peak gate on the full clip
        from scipy.signal import savgol_filter
        cap2 = cv2.VideoCapture(str(video_path))
        energies, prev_gray = [], None
        for fi in range(0, total, 2):
            cap2.set(cv2.CAP_PROP_POS_FRAMES, fi)
            ret, frame = cap2.read()
            if not ret: break
            gray = cv2.GaussianBlur(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY),(5,5),0)
            if prev_gray is not None:
                energies.append((fi, float(np.mean(cv2.absdiff(gray, prev_gray)))))
            prev_gray = gray
        cap2.release()
        if energies:
            vals  = np.array([e[1] for e in energies])
            idxs  = np.array([e[0] for e in energies])
            win   = min(15, len(vals)-(1-len(vals)%2)); win=max(win if win%2==1 else win-1,5)
            sm    = savgol_filter(vals, win, 3) if len(vals)>win else vals
            n     = len(sm); gated=sm.copy()
            gated[:int(n*0.30)]=0; gated[int(n*0.85):]=0
            onset = int(idxs[int(np.argmax(gated))])
        else:
            return None

    start_s = max(0.0, onset / fps - PRE_SWING_S)
    end_s   = min(dur_s, onset / fps + POST_SWING_S)

    if end_s - start_s < MIN_CLIP_S:
        return None   # too short — something went wrong, skip

    # Use ffmpeg for lossless stream-copy trim (fast, no re-encode)
    if shutil.which('ffmpeg'):
        result = subprocess.run([
            'ffmpeg', '-y',
            '-ss', f'{start_s:.3f}',
            '-to', f'{end_s:.3f}',
            '-i', str(video_path),
            '-c', 'copy',
            '-avoid_negative_ts', 'make_zero',
            str(out_path),
        ], capture_output=True)
        if result.returncode != 0:
            return None
    else:
        # OpenCV fallback
        cap   = cv2.VideoCapture(str(video_path))
        w_px  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h_px  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        writer = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*'mp4v'), fps, (w_px, h_px))
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(start_s * fps))
        for _ in range(int((end_s - start_s) * fps)):
            ret, frame = cap.read()
            if not ret: break
            writer.write(frame)
        cap.release(); writer.release()

    return out_path


# ── Run trim on all videos ────────────────────────────────────────────────────
trimmed_videos = []
trim_failed    = []

for vp in tqdm(all_videos, desc='Trimming (hip+arm gate)'):
    try:
        out = trim_to_swing(vp)
        if out is not None:
            trimmed_videos.append(out)
        else:
            print(f'  SKIP {vp.name} (no onset detected or clip too short)')
            trim_failed.append(vp.name)
    except Exception as e:
        print(f'  FAILED {vp.name}: {e}')
        trim_failed.append(vp.name)

print(f'\nTrimmed: {len(trimmed_videos)} | Failed/skipped: {len(trim_failed)}')

# Replace all_videos with the trimmed versions for downstream cells
all_videos_original = all_videos
all_videos = trimmed_videos
print(f'all_videos now points to {len(all_videos)} trimmed clips')
print(f'Example: {all_videos[0].name if all_videos else "none"}')

Trimming (hip+arm gate):  82%|████████▏ | 983/1197 [28:18<30:09,  8.46s/it]

  SKIP Alec Bohm.mp4 (no onset detected or clip too short)


Trimming (hip+arm gate):  84%|████████▎ | 1001/1197 [29:03<14:40,  4.49s/it]

  SKIP Bobby Witt Jr_swing.mp4 (no onset detected or clip too short)


Trimming (hip+arm gate):  88%|████████▊ | 1051/1197 [30:50<07:51,  3.23s/it]

  SKIP J.D. Martinez_swing.mp4 (no onset detected or clip too short)


Trimming (hip+arm gate): 100%|██████████| 1197/1197 [35:45<00:00,  1.79s/it]


Trimmed: 1194 | Failed/skipped: 3
all_videos now points to 1194 trimmed clips
Example: Aaron Judge_swing.mp4


## 2c. Delete `_swing.mp4` duplicates from Drive

Each time the trim cell runs it writes a `<stem>_swing.mp4` alongside the original.
Run this cell to remove all `_swing.mp4` files and keep only the originals.

> **Only run this after you are happy with the current trim results.**
> Set `CONFIRM_DELETE = True` to actually delete (dry-run by default).

In [ ]:
from pathlib import Path
from tqdm import tqdm

CONFIRM_DELETE = True   # ← set to True to actually delete

swing_files = sorted(
    p for p in DRIVE_VIDEO_DIR.rglob('*')
    if p.is_file()
    and p.stem.endswith('_swing')
    and p.suffix.lower() in {'.mp4', '.mov', '.avi', '.mkv'}
)

print(f"Found {len(swing_files)} _swing file(s):")
for f in swing_files:
    print(f"  {f.relative_to(DRIVE_VIDEO_DIR)}  ({f.stat().st_size/1e6:.1f} MB)")

if not swing_files:
    print("Nothing to delete.")
elif not CONFIRM_DELETE:
    total_mb = sum(f.stat().st_size for f in swing_files) / 1e6
    print(f"\nDry run — {total_mb:.0f} MB would be freed.")
    print("Set CONFIRM_DELETE = True to delete.")
else:
    deleted = 0
    for f in tqdm(swing_files, desc="Deleting"):
        try:
            f.unlink()
            deleted += 1
        except Exception as e:
            print(f"  Could not delete {f.name}: {e}")
    freed_mb = sum(0 for _ in [])   # already deleted, can't stat
    print(f"\nDeleted {deleted}/{len(swing_files)} _swing file(s).")

    # Refresh all_videos so downstream cells don't reference deleted files
    VIDEO_EXTS = {'.mp4', '.mov', '.avi', '.mkv'}
    all_videos = sorted(
        p for p in DRIVE_VIDEO_DIR.rglob('*')
        if p.is_file()
        and p.suffix.lower() in VIDEO_EXTS
        and '_annotated' not in p.stem
        and '_swing'     not in p.stem
        and 'outputs'    not in str(p)
    )
    print(f"all_videos refreshed: {len(all_videos)} original clips remaining.")

Found 1194 _swing file(s):
  6s_trimmed_videos/Aaron Judge_swing.mp4  (1.6 MB)
  6s_trimmed_videos/Aaron Judge_swing_swing.mp4  (1.2 MB)
  6s_trimmed_videos/Aaron Judge_swing_swing_swing.mp4  (1.2 MB)
  6s_trimmed_videos/Aaron Judge_swing_swing_swing_swing.mp4  (1.2 MB)
  6s_trimmed_videos/Adam Duvall_swing.mp4  (2.3 MB)
  6s_trimmed_videos/Adam Duvall_swing_swing.mp4  (1.6 MB)
  6s_trimmed_videos/Adam Duvall_swing_swing_swing.mp4  (1.6 MB)
  6s_trimmed_videos/Adam Duvall_swing_swing_swing_swing.mp4  (1.6 MB)
  6s_trimmed_videos/Adley Rutschman_swing.mp4  (1.2 MB)
  6s_trimmed_videos/Adley Rutschman_swing_swing.mp4  (1.2 MB)
  6s_trimmed_videos/Adley Rutschman_swing_swing_swing.mp4  (1.2 MB)
  6s_trimmed_videos/Adley Rutschman_swing_swing_swing_swing.mp4  (1.2 MB)
  6s_trimmed_videos/Adolis García_swing.mp4  (2.5 MB)
  6s_trimmed_videos/Adolis García_swing_swing.mp4  (2.5 MB)
  6s_trimmed_videos/Adolis García_swing_swing_swing.mp4  (2.5 MB)
  6s_trimmed_videos/Adolis García_swing_s

Deleting: 100%|██████████| 1194/1194 [00:02<00:00, 403.52it/s]



Deleted 1194/1194 _swing file(s).
all_videos refreshed: 424 original clips remaining.


## 5b. Clear existing YOLO sidecars ← run before re-running YOLO

Deletes all `.yolo.json` files from the video folder so YOLO runs fresh on every video.
**Run this cell only when you want a clean YOLO re-run** (e.g. after trimming videos
or after changing YOLO parameters). Skip it if you want to keep existing sidecars.

In [ ]:
from pathlib import Path
from tqdm import tqdm

# Find all .yolo.json files in the video folder (same locations as the videos)
sidecar_paths = sorted(
    p for v in all_videos
    for p in [v.with_suffix('.yolo.json')]
    if p.exists()
)

# Also catch any orphaned sidecars in the Drive root
sidecar_paths += [
    p for p in DRIVE_VIDEO_DIR.rglob('*.yolo.json')
    if p not in sidecar_paths
]

print(f'Found {len(sidecar_paths)} .yolo.json sidecar(s) to delete.')
for sp in sidecar_paths:
    print(f'  {sp.relative_to(DRIVE_VIDEO_DIR)}')

# Confirmation guard — set CONFIRM_DELETE = True to actually delete
CONFIRM_DELETE = True   # ← flip to True to delete

if not CONFIRM_DELETE:
    print('\nSet CONFIRM_DELETE = True in this cell to actually delete the files.')
else:
    deleted = 0
    for sp in tqdm(sidecar_paths, desc='Deleting sidecars'):
        try:
            sp.unlink()
            deleted += 1
        except Exception as e:
            print(f'  Could not delete {sp.name}: {e}')
    print(f'\nDeleted {deleted}/{len(sidecar_paths)} sidecar(s).')
    print('Run the YOLO cell now to regenerate them.')

Found 506 .yolo.json sidecar(s) to delete.
  Aaron Judge. Big swing in slow motion._swing.yolo.json
  Adam Duvall. Big swing in slow motion._swing.yolo.json
  Adley Rutschman. Big swing in slow motion._swing.yolo.json
  Adolis García. Big swing in slow motion._swing.yolo.json
  Akin Baddoo. Big swing in slow motion._swing.yolo.json
  Alec Bohm. Big swing in slow motion._swing.yolo.json
  Alejandro Kirk. Big swing in slow motion._swing.yolo.json
  Alex Bregman. Big swing in slow motion._swing.yolo.json
  Andrew Vaughn. Big swing in slow motion._swing.yolo.json
  Anthony Rendon. Big swing in slow motion._swing.yolo.json
  Anthony Volpe. Big swing in slow motion._swing.yolo.json
  Austin Riley. Big swing in slow motion._swing.yolo.json
  Bo Bichette . Big swing in slow motion._swing.yolo.json
  Bobby Dalbec. Big swing in slow motion._swing.yolo.json
  Bobby Witt Jr. Big swing in slow motion._swing.yolo.json
  Bryan Reynolds. Big swing in slow motion._swing.yolo.json
  Bryce Harper. Big s

Deleting sidecars: 100%|██████████| 506/506 [00:01<00:00, 503.33it/s]


Deleted 506/506 sidecar(s).
Run the YOLO cell now to regenerate them.


## 5. Stage 1 — YOLO sidecars

**Base YOLO (`yolov8n.pt`) cannot reliably detect baseballs** at broadcast resolution.
The COCO "sports ball" class was trained on soccer/basketball/volleyball — all 5-10× larger
than a baseball. Running it produces R²=0.000 on every video (random noise, not a ball arc).

**Current approach — empty sidecars (instant):**
Write a minimal `.yolo.json` next to every video so MediaPipe can proceed.
`ball_angle` and `ball_contact_px` are set to `null`; the CNN trains on pose-only angles.
This is the correct path until YOLO is fine-tuned on labelled baseball data.

**When YOLO is fine-tuned** (Roboflow Universe has labelled baseball datasets):
Uncomment the original YOLO block at the bottom of the code cell below.
Fine-tuning takes ~50 epochs; point `YOLO_WEIGHTS` at `runs/detect/train/weights/best.pt`.

In [ ]:
import cv2, json
from tqdm import tqdm

# ── Write empty sidecars so MediaPipe can proceed without ball tracking ───────
# This is the correct approach when base YOLO weights fail to detect the baseball.
# Empty sidecars tell MediaPipe: no ball data available, use pose-only angles.

written = 0
skipped = 0
for vp in tqdm(all_videos, desc='Writing sidecars'):
    sp = vp.with_suffix('.yolo.json')
    if sp.exists():
        skipped += 1
        continue
    cap   = cv2.VideoCapture(str(vp))
    fps   = cap.get(cv2.CAP_PROP_FPS) or 60.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    with open(sp, 'w') as f:
        json.dump({
            "video_path":       str(vp),
            "fps":              fps,
            "total_frames":     total,
            "clip_start_frame": 0,
            "contact_frame":    int(total * 0.50),   # rough midpoint fallback
            "contact_time_s":   round(total * 0.50 / fps, 3),
            "ball_angle":       None,
            "ball_contact_px":  None,
            "post_contact":     [],
            "pre_contact":      [],
            "bat_boxes":        {}
        }, f)
    written += 1

print(f'Sidecars written: {written}  |  Already existed: {skipped}  |  Total: {written+skipped}/{len(all_videos)}')
print('Proceeding with pose-only angles (ball_angle=null for all videos).')

# ════════════════════════════════════════════════════════════════════════════════
# OPTIONAL — uncomment this block once you have fine-tuned YOLO weights
# ════════════════════════════════════════════════════════════════════════════════
# Fine-tuning instructions:
#   1. Download labelled baseball data from Roboflow Universe:
#        https://universe.roboflow.com/search?q=baseball+bat+ball
#      Export in YOLOv8 format.
#   2. Create data.yaml with: nc=2, names=[ball, bat]
#   3. Fine-tune (run in yolo_env):
#        from ultralytics import YOLO
#        model = YOLO('yolov8n.pt')
#        model.train(data='data.yaml', epochs=50, imgsz=640, batch=8)
#   4. Set YOLO_WEIGHTS below to 'runs/detect/train/weights/best.pt'
#      and uncomment the block.
#
# import sys
# from pathlib import Path
# sys.path.insert(0, '/content')
# import yolo_processing as yolo_mod
#
# YOLO_WEIGHTS = 'runs/detect/train/weights/best.pt'   # ← your fine-tuned weights
# yolo_mod.PITCHER_SIDE       = 'L'
# yolo_mod.POST_SEARCH_FRAMES = 60
# yolo_mod.PRE_SEARCH_FRAMES  = 40
# yolo_mod.VIDEO_ROOT         = DRIVE_VIDEO_DIR
#
# try:
#     detector = yolo_mod.YOLODetector(YOLO_WEIGHTS)
#     print(f'Fine-tuned YOLO loaded: {YOLO_WEIGHTS}')
# except Exception as e:
#     detector = None
#     print(f'YOLO load failed ({e})')
#
# to_process = [v for v in all_videos if not v.with_suffix('.yolo.json').exists()]
# print(f'Running YOLO on {len(to_process)} videos...')
# failed_yolo = []
# for vp in tqdm(to_process, desc='YOLO'):
#     try:
#         result = yolo_mod.process_video(vp, detector)
#         with open(vp.with_suffix('.yolo.json'), 'w') as f:
#             json.dump(result, f, indent=2)
#     except Exception as e:
#         print(f'  FAILED {vp.name}: {e}')
#         failed_yolo.append(vp.name)
# done = sum(1 for v in all_videos if v.with_suffix('.yolo.json').exists())
# print(f'Sidecars: {done}/{len(all_videos)}  |  Failed: {len(failed_yolo)}')

Writing sidecars: 100%|██████████| 424/424 [02:43<00:00,  2.59it/s]

Sidecars written: 413  |  Already existed: 11  |  Total: 424/424
Proceeding with pose-only angles (ball_angle=null for all videos).


## 6. Stage 2 — MediaPipe pose estimation + launch angle

**Speed optimisations (5 hrs → ~35 min for 434 × 6s clips):**
- **Stride 6** — pose runs at 10fps instead of 30fps; sufficient for wrist deceleration detection
- **YOLO-anchored window** — only process ±54 frames around the YOLO contact estimate instead of the full clip; cuts frames processed by ~70%
- **PoseLandmarker Lite** — ~3× faster inference than Heavy; accurate enough for wrist/elbow localisation
- **30–85% gate fallback** — used when no YOLO sidecar is available

Hardcoded: all batters treated as **right-handed** (`HANDEDNESS = "R"`).

In [ ]:
import sys, json, csv
from pathlib import Path
from dataclasses import asdict
from tqdm import tqdm

sys.path.insert(0, '/content')
import mediapipe_processing as mp_mod

# --- Start of Patch to handle malformed .yolo.json sidecars ---
import logging
from typing import Optional # Import Optional for type hinting

def patched_load_sidecar(video_path: Path) -> Optional[dict]:
    sidecar_path_obj = video_path.with_suffix(".yolo.json")
    if not sidecar_path_obj.exists():
        logging.warning(
            "No sidecar found for %s — ball tracking disabled for this video.\n"
            "  Run yolo_processing.py first to generate %s",
            video_path.name, sidecar_path_obj.name,
        )
        return None
    try:
        with open(sidecar_path_obj) as f:
            return json.load(f)
    except json.JSONDecodeError as e:
        logging.warning(
            "Malformed sidecar found for %s: %s — ball tracking disabled for this video.",
            video_path.name, e,
        )
        return None

# Assign the patched function back to the module
mp_mod.load_sidecar = patched_load_sidecar
# --- End of Patch ---


# Override paths
mp_mod.VIDEO_ROOT  = DRIVE_VIDEO_DIR
mp_mod.OUTPUT_DIR  = OUTPUT_DIR
mp_mod.BATCH_CSV   = OUTPUT_DIR / 'batch_results.csv'
mp_mod.MODEL_PATH  = MODEL_PATH
mp_mod.PROOF_CHECK = False
mp_mod.STRIDE             = 6
mp_mod.YOLO_ANCHOR_WINDOW = 54   # ±54 frames around YOLO contact estimate

print(f'Analysing {len(all_videos)} videos with MediaPipe...')

swing_results = []   # list of (SwingResult, frame_results, video_path, sidecar)
failed_mp     = []

for vp in tqdm(all_videos, desc='MediaPipe'):
    sidecar    = mp_mod.load_sidecar(vp)
    landmarker = mp_mod.load_pose_landmarker(mp_mod.MODEL_PATH)
    try:
        result, frame_results = mp_mod.analyze_swing(
            vp, landmarker, sidecar=sidecar, verbose=False
        )
        # Force handedness to R for this dataset
        result.batter_handedness = 'R'
        swing_results.append((result, frame_results, vp, sidecar))
        json_out = OUTPUT_DIR / (vp.stem + '_result.json')
        with open(json_out, 'w') as f:
            json.dump(asdict(result), f, indent=2)
    except Exception as e:
        print(f'  FAILED {vp.name}: {e}')
        failed_mp.append(vp.name)
    finally:
        landmarker.close()

if swing_results:
    fieldnames = list(asdict(swing_results[0][0]).keys())
    with open(mp_mod.BATCH_CSV, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        for r,_,_,_ in swing_results:
            row = asdict(r)
            for lf in ('bat_vector','ball_centroids','pre_contact_centroids'):
                row[lf] = json.dumps(row[lf])
            w.writerow(row)
    print(f'Batch CSV → {mp_mod.BATCH_CSV}')

print(f'Done: {len(swing_results)} processed, {len(failed_mp)} failed.')

## 6b. Inspect one MediaPipe training sample

Pulls the first entry from `swing_results`, displays:
1. **The contact-frame patch** — the 256×256 crop the CNN actually sees (all 3 temporal frames side-by-side)
2. **The raw label** — the `launch_angle` value in degrees that the CNN is trained to predict
3. **A breakdown of how that label was computed** — image-space angle, world-space angle, and the 70/30 blend
4. **The full `_result.json`** — every field MediaPipe stored for this video

This lets you verify that the labels are geometrically sensible before training.

In [ ]:
import cv2, json, math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from dataclasses import asdict

# ── Pick the first successfully processed video ───────────────────────────────
if not swing_results:
    print("swing_results is empty — run the MediaPipe cell (or restore session) first.")
else:
    result, frame_results, vp, sidecar = swing_results[0]
    print(f"Video: {vp.name}")

    # ── 1. Load the stored label ──────────────────────────────────────────────
    label = result.launch_angle
    rj    = OUTPUT_DIR / (vp.stem + '_result.json')
    with open(rj) as f:
        stored = json.load(f)

    # ── 2. Extract contact-frame patch (all 3 temporal frames) ───────────────
    lms_map = {fi: lms for fi, lms in frame_results}
    cf      = result.contact_frame
    lms_cf  = lms_map.get(cf)

    cap   = cv2.VideoCapture(str(vp))
    fps   = cap.get(cv2.CAP_PROP_FPS) or 60.0
    w_px  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h_px  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Wrist crop centre from landmarks (image-space offset 33)
    cx, cy = w_px // 2, h_px // 2
    if lms_cf and len(lms_cf) >= 66:
        import mediapipe_processing as mp_mod
        OFF = 33
        lw  = lms_cf[OFF + mp_mod.LM['l_wrist']]
        rw  = lms_cf[OFF + mp_mod.LM['r_wrist']]
        cx  = int(((lw.x + rw.x) / 2) * w_px)
        cy  = int(((lw.y + rw.y) / 2) * h_px)

    PATCH = 256
    def get_patch(fi):
        cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, min(total-1, fi)))
        ret, frame = cap.read()
        if not ret:
            return np.zeros((PATCH, PATCH, 3), dtype=np.uint8)
        h, w = frame.shape[:2]
        half = PATCH // 2
        x1, y1 = cx - half, cy - half
        x2, y2 = cx + half, cy + half
        pl=max(0,-x1); pr=max(0,x2-w); pt=max(0,-y1); pb=max(0,y2-h)
        crop = frame[max(0,y1):min(h,y2), max(0,x1):min(w,x2)]
        if pl or pr or pt or pb:
            crop = cv2.copyMakeBorder(crop, pt,pb,pl,pr, cv2.BORDER_CONSTANT, value=0)
        patch = cv2.resize(crop, (PATCH, PATCH))
        return cv2.cvtColor(patch, cv2.COLOR_BGR2RGB)

    patch_tm2 = get_patch(cf - 2)
    patch_t0  = get_patch(cf)
    patch_tp2 = get_patch(cf + 2)
    cap.release()

    # ── 3. Plot ───────────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(15, 9))
    fig.patch.set_facecolor('#0f0f1a')

    # ── Row 1: three temporal patches ─────────────────────────────────────────
    titles    = ['t − 2  (incoming bat)', 't = 0  (CONTACT FRAME)', 't + 2  (departure)']
    patches_3 = [patch_tm2, patch_t0, patch_tp2]
    colors    = ['#4a9eff', '#ff6b35', '#4a9eff']

    for col, (patch, title, border_col) in enumerate(zip(patches_3, titles, colors)):
        ax = fig.add_subplot(2, 4, col + 1)
        ax.imshow(patch)
        ax.set_title(title, color='white', fontsize=10, pad=6)
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_edgecolor(border_col); spine.set_linewidth(2.5)
        # Draw wrist crosshair on each patch
        cx_p = PATCH // 2; cy_p = PATCH // 2
        gap = 18; arm = 36
        ax.plot([cx_p-arm, cx_p-gap], [cy_p, cy_p], color=border_col, lw=1.5)
        ax.plot([cx_p+gap, cx_p+arm], [cy_p, cy_p], color=border_col, lw=1.5)
        ax.plot([cx_p, cx_p], [cy_p-arm, cy_p-gap], color=border_col, lw=1.5)
        ax.plot([cx_p, cx_p], [cy_p+gap, cy_p+arm], color=border_col, lw=1.5)
        ax.set_facecolor('#0f0f1a')

    # ── Panel 4: angle diagram ─────────────────────────────────────────────────
    ax4 = fig.add_subplot(2, 4, 4)
    ax4.set_facecolor('#1a1a2e')
    ax4.set_xlim(-1.3, 1.3); ax4.set_ylim(-1.3, 1.3)
    ax4.set_aspect('equal'); ax4.axis('off')
    ax4.set_title('Launch angle geometry', color='white', fontsize=10, pad=6)

    # Horizontal reference
    ax4.annotate('', xy=(1.1, 0), xytext=(-1.1, 0),
                 arrowprops=dict(arrowstyle='->', color='#555', lw=1.2))
    ax4.text(1.15, 0, '→ pitcher', color='#777', fontsize=8, va='center')

    # Launch angle arc
    angle_rad = math.radians(label)
    ax4.annotate('', xy=(math.cos(angle_rad)*0.9, math.sin(angle_rad)*0.9),
                 xytext=(0, 0),
                 arrowprops=dict(arrowstyle='->', color='#ff6b35', lw=2.5))
    ax4.text(math.cos(angle_rad)*1.05, math.sin(angle_rad)*1.05,
             f'{label:+.1f}°', color='#ff6b35', fontsize=12, fontweight='bold',
             ha='center', va='center')

    # Arc sweep
    arc_angles = np.linspace(0, angle_rad, 40)
    ax4.plot(np.cos(arc_angles)*0.35, np.sin(arc_angles)*0.35,
             color='#4a9eff', lw=1.5, linestyle='--')

    # Zone label
    ANGLE_ZONES = [
        (-90,-10,'Ground ball'),(-10,5,'Hard liner'),(5,15,'Line drive'),
        (15,25,'Solid fly'),(25,35,'HR window'),(35,50,'High fly'),(50,90,'Pop-up'),
    ]
    zone_label = next((l for lo,hi,l in ANGLE_ZONES if lo<=label<hi),
                      'Pop-up' if label>=50 else 'Ground ball')
    ax4.text(0, -1.2, f'Zone: {zone_label}', color='#2ecc71',
             fontsize=9, ha='center', fontweight='bold')
    ax4.text(0, 1.25, 'Bat vector at contact', color='#aaa', fontsize=8, ha='center')

    for spine in ax4.spines.values():
        spine.set_edgecolor('#333'); spine.set_linewidth(1)

    # ── Row 2: label breakdown table ──────────────────────────────────────────
    ax_table = fig.add_subplot(2, 1, 2)
    ax_table.set_facecolor('#0f0f1a')
    ax_table.axis('off')

    rows = [
        ['Field', 'Value', 'What it means'],
        ['video',              vp.stem[:40],                'Source video filename'],
        ['contact_frame',      str(cf),                     f'Frame index of bat-ball contact  (t={cf/fps:.2f}s)'],
        ['contact_time_s',     f"{result.contact_time_s:.3f}s",   'Timestamp of contact in the clip'],
        ['batter_handedness',  result.batter_handedness,   'Inferred from shoulder Z-depth (front shoulder closer to camera)'],
        ['confidence',         f"{result.confidence:.0%}",  'Gaussian-weighted landmark visibility ±10 frames around contact'],
        ['launch_angle (LABEL)', f"{label:+.2f}°",         '70% image-space + 30% world-space bat vector angle — CNN target'],
        ['contact_zone',       result.contact_zone,         '9-cell strike zone grid (High/Mid/Low × In/Middle/Out)'],
        ['landing_zone',       result.landing_zone,         'Rule-table lookup: angle → predicted field region'],
        ['ball_angle',         str(result.ball_angle),      'From YOLO ball tracking — null (YOLO not yet fine-tuned)'],
    ]

    col_x    = [0.01, 0.30, 0.55]
    row_h    = 0.085
    header   = rows[0]
    data_rows = rows[1:]

    for c, (hdr, x) in enumerate(zip(header, col_x)):
        ax_table.text(x, 1.0, hdr, color='#4a9eff', fontsize=9,
                      fontweight='bold', transform=ax_table.transAxes, va='top')

    for r, row in enumerate(data_rows):
        bg_col = '#1a1a2e' if r % 2 == 0 else '#12121f'
        ax_table.add_patch(mpatches.FancyBboxPatch(
            (0, 1.0 - (r+1)*row_h - 0.01), 1.0, row_h - 0.005,
            boxstyle='round,pad=0.005', facecolor=bg_col,
            transform=ax_table.transAxes, zorder=0, linewidth=0
        ))
        highlight = r == 5   # launch_angle row
        for c, (val, x) in enumerate(zip(row, col_x)):
            color = '#ff6b35' if highlight else ('#ddd' if c == 0 else '#bbb')
            ax_table.text(x, 1.0 - (r+1)*row_h + row_h*0.3, val,
                          color=color, fontsize=8.5,
                          transform=ax_table.transAxes, va='center',
                          fontweight='bold' if highlight else 'normal')

    ax_table.set_title(
        f'MediaPipe training label — {vp.stem[:50]}',
        color='white', fontsize=11, pad=8, loc='left'
    )

    plt.suptitle('MediaPipe Sample Annotation', color='white', fontsize=14,
                 fontweight='bold', y=1.01)
    plt.tight_layout(rect=[0, 0, 1, 1])
    out_img = OUTPUT_DIR / 'mediapipe_sample_annotation.png'
    plt.savefig(str(out_img), dpi=130, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.show()

    # ── 4. Print the raw label dict ───────────────────────────────────────────
    print(f"\n{'─'*55}")
    print(f"Raw label stored in {vp.stem}_result.json")
    print(f"{'─'*55}")
    for k, v in stored.items():
        if k in ('bat_vector', 'ball_centroids', 'pre_contact_centroids'):
            v = f'[list of {len(json.loads(v)) if isinstance(v,str) else len(v)} items]'
        print(f"  {k:<28s}: {v}")
    print(f"\nAnnotation image saved → {out_img}")

## 7. Stage 3 — Annotate all videos

In [ ]:
from tqdm import tqdm
import cv2, math, numpy as np

# ── Patch draw_frame to add contact flash and gate angle display ──────────────
# We wrap mp_mod.draw_frame so the contact frame gets a full-frame flash,
# and the launch angle ray + HUD only appear AT or AFTER the contact frame.

import mediapipe_processing as _mp

_orig_draw_frame = _mp.draw_frame

def _patched_draw_frame(
    frame, lms, result, frame_idx, post_contact,
    h, w, ball_trail, pre_ball, contact_px, bat_bbox, sidecar
):
    # 1. Call the original draw_frame — but suppress angle ray by temporarily
    #    nulling ball_angle when we're before contact
    if not post_contact:
        import copy
        _r = copy.copy(result)
        _r.ball_angle = None          # hides the angle ray before contact
        out = _orig_draw_frame(
            frame, lms, _r, frame_idx, post_contact,
            h, w, ball_trail, pre_ball, contact_px, bat_bbox, sidecar
        )
    else:
        out = _orig_draw_frame(
            frame, lms, result, frame_idx, post_contact,
            h, w, ball_trail, pre_ball, contact_px, bat_bbox, sidecar
        )

    cf  = result.contact_frame
    age = frame_idx - cf   # negative = pre-contact, 0 = contact frame, positive = post

    # 2. Contact-frame flash — bright white vignette on the exact contact frame
    if age == 0:
        flash = out.copy()
        flash[:] = (255, 255, 255)
        cv2.addWeighted(flash, 0.35, out, 0.65, 0, out)
        # Bold "CONTACT" banner at top-centre
        font  = cv2.FONT_HERSHEY_SIMPLEX
        label = ">>> CONTACT FRAME <<<"
        tw, th = cv2.getTextSize(label, font, 1.1, 3)[0]
        tx = (w - tw) // 2
        # Drop shadow
        cv2.putText(out, label, (tx+2, 52), font, 1.1, (0, 0, 0),    4, cv2.LINE_AA)
        cv2.putText(out, label, (tx,   50), font, 1.1, (0, 220, 255), 3, cv2.LINE_AA)

    # 3. Persistent contact marker — pulsing rings that fade over 40 frames
    if post_contact and contact_px is not None:
        cx_c, cy_c = contact_px
        alpha = max(0.0, 1.0 - age / 40.0)
        if alpha > 0:
            for rr in [14, 26, 42]:
                ov = out.copy()
                cv2.circle(ov, (cx_c, cy_c), rr, (0, 80, 255), 2, cv2.LINE_AA)
                cv2.addWeighted(ov, alpha, out, 1 - alpha, 0, out)
        cv2.circle(out, (cx_c, cy_c), 6, (0, 80, 255), -1, cv2.LINE_AA)

    # 4. Launch angle ray and label — ONLY at or after contact
    if post_contact and result.launch_angle is not None and contact_px is not None:
        cx_c, cy_c = contact_px
        ang_rad = math.radians(result.launch_angle)
        ray_len = 220
        ex = int(cx_c + ray_len * math.cos(ang_rad))
        ey = int(cy_c - ray_len * math.sin(ang_rad))
        cv2.arrowedLine(out, (cx_c, cy_c), (ex, ey), (255, 200, 0), 2, cv2.LINE_AA, tipLength=0.10)
        angle_label = f"{result.launch_angle:+.1f} deg"
        cv2.putText(out, angle_label, (ex + 6, ey),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 200, 0), 1, cv2.LINE_AA)

    # 5. HUD — suppress launch angle line before contact, show [CNN-validated] after
    if post_contact:
        ov = out.copy()
        cv2.rectangle(ov, (10, 10), (380, 130), (18, 18, 28), -1)
        cv2.addWeighted(ov, 0.65, out, 0.35, 0, out)
        cv2.rectangle(out, (10, 10), (380, 130), (70, 70, 70), 1)
        font = cv2.FONT_HERSHEY_SIMPLEX
        cf_src = "ball" if result.ball_contact_px else "pose"
        cv2.putText(out, f"** CONTACT  t={result.contact_time_s:.2f}s  [{cf_src}]",
                    (20, 35),  font, 0.52, (0, 220, 180), 1, cv2.LINE_AA)
        cv2.putText(out, f"Launch angle : {result.launch_angle:+.1f} deg",
                    (20, 57),  font, 0.52, (220, 220, 220), 1, cv2.LINE_AA)
        # Zone
        ANGLE_ZONES = [
            (-90,-10,'Ground ball'),(-10,5,'Hard liner'),(5,15,'Line drive'),
            (15,25,'Solid fly'),(25,35,'HR window'),(35,50,'High fly'),(50,90,'Pop-up'),
        ]
        zone = next((l for lo,hi,l in ANGLE_ZONES if lo<=result.launch_angle<hi),
                    'Pop-up' if result.launch_angle>=50 else 'Ground ball')
        cv2.putText(out, f"Landing zone : {zone}",
                    (20, 79),  font, 0.52, (220, 220, 220), 1, cv2.LINE_AA)
        cv2.putText(out, f"Contact zone : {result.contact_zone}",
                    (20, 101), font, 0.52, (220, 220, 220), 1, cv2.LINE_AA)
        cv2.putText(out,
                    f"Hand: {result.batter_handedness}   Conf: {result.confidence:.0%}   [MediaPipe]",
                    (20, 123), font, 0.44, (140, 140, 140), 1, cv2.LINE_AA)

    return out

# Monkey-patch so write_annotated_video picks it up
_mp.draw_frame = _patched_draw_frame

# ── Run annotation ────────────────────────────────────────────────────────────
ann_done, ann_failed = [], []
for result, frame_results, vp, sidecar in tqdm(swing_results, desc='Annotating'):
    out_path = vp.parent / (vp.stem + '_annotated.mp4')
    if out_path.exists():
        ann_done.append(out_path.name)
        continue
    try:
        _mp.write_annotated_video(
            vp, result, frame_results, out_path, sidecar, contact_window=120
        )
        ann_done.append(out_path.name)
    except Exception as e:
        print(f'  FAILED {vp.name}: {e}')
        ann_failed.append(vp.name)

# Restore original so other cells aren't affected
_mp.draw_frame = _orig_draw_frame

print(f'Annotated: {len(ann_done)}, Failed: {len(ann_failed)}')

## 8. Stage 4 — Extract 3-frame temporal CNN dataset

**Why 3 frames instead of 1:**
The single-frame patch captures *where* the bat is but not *how fast* or *at what angle* it is moving. Launch angle depends critically on bat speed and attack angle at the moment of contact — information that lives in the *difference* between frames, not in a single frame.

By stacking frames at `t-2, t, t+2` (where `t` = contact frame) as a **9-channel input** (3 RGB frames × 3 channels), the CNN receives:
- The bat position at contact (`t`)
- The bat's incoming trajectory (`t-2 → t`)
- The ball's initial departure (`t → t+2`)

This encodes the full momentum transfer event in one forward pass.

**Patch strategy:**
- **256×256 crop** centred on the wrist midpoint (larger than before to capture full bat arc)
- Saved as JPG for `t`, `t-2`, `t+2` frames separately (stacked at load time)
- 80/20 train/val split

In [ ]:
import cv2, json, random, numpy as np
from pathlib import Path
from tqdm import tqdm
from dataclasses import asdict
import mediapipe_processing as mp_mod

PATCH_SIZE = 256
VAL_FRAC   = 0.20
random.seed(42)

def extract_patch(frame, cx, cy, size):
    h, w = frame.shape[:2]
    half = size // 2
    x1,y1,x2,y2 = cx-half,cy-half,cx+half,cy+half
    pl = max(0,-x1); pr = max(0,x2-w)
    pt = max(0,-y1); pb = max(0,y2-h)
    crop = frame[max(0,y1):min(h,y2), max(0,x1):min(w,x2)]
    if pl or pr or pt or pb:
        crop = cv2.copyMakeBorder(crop, pt,pb,pl,pr, cv2.BORDER_CONSTANT, value=0)
    return cv2.resize(crop, (size, size))

samples = []
skipped = 0
LM = mp_mod.LM

for result, frame_results, vp, sidecar in tqdm(swing_results, desc='Extracting'):
    if result.confidence < 0.55:
        skipped += 1; continue

    lms_map = {fi: lms for fi, lms in frame_results}
    cf = result.contact_frame

    # Need landmarks at contact frame for wrist position
    lms = lms_map.get(cf)
    if lms is None or len(lms) < 66:
        skipped += 1; continue

    OFF = 33
    lw  = lms[OFF + LM['l_wrist']]
    rw  = lms[OFF + LM['r_wrist']]

    cap   = cv2.VideoCapture(str(vp))
    w_px  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h_px  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    cx = int(((lw.x + rw.x) / 2) * w_px)
    cy = int(((lw.y + rw.y) / 2) * h_px)

    patches = {}
    for offset, tag in [(-2,'tm2'), (0,'t0'), (2,'tp2')]:
        fi = max(0, min(total-1, cf + offset))
        cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
        ret, frame = cap.read()
        if ret:
            patches[tag] = extract_patch(frame, cx, cy, PATCH_SIZE)
    cap.release()

    if len(patches) < 3:
        skipped += 1; continue

    samples.append({
        'stem':         vp.stem,
        'patches':      patches,          # dict of 3 BGR images
        'launch_angle': result.launch_angle,
        'confidence':   result.confidence,
        'contact_frame': cf,
    })

print(f'Samples collected: {len(samples)}, Skipped: {skipped}')

# 80/20 split
random.shuffle(samples)
n_val     = max(1, int(len(samples) * VAL_FRAC))
val_set   = samples[:n_val]
train_set = samples[n_val:]

for split_name, split_data in [('train', train_set), ('val', val_set)]:
    split_dir = DATASET_DIR / split_name
    split_dir.mkdir(parents=True, exist_ok=True)
    for s in split_data:
        stem = s['stem']
        # Save three frames separately; they're stacked at load time
        for tag, patch in s['patches'].items():
            cv2.imwrite(str(split_dir / f"{stem}_{tag}.jpg"), patch)
        with open(split_dir / f"{stem}_label.json", 'w') as f:
            json.dump({'launch_angle': s['launch_angle'],
                       'confidence':  s['confidence']}, f)
    print(f"  {split_name}: {len(split_data)} samples → {split_dir}")

## 9. Stage 5 — CNN architecture

**Model: ResNet-18 with 9-channel temporal input**

The first convolution layer is modified from the standard 3-channel (RGB) to **9-channel** to accept the 3-frame stack (t-2, t, t+2). The ImageNet pretrained weights for the first layer are averaged across the 3 input channel groups so no information is discarded.

**Augmentation designed for right-angle geometry:**
- `RandomAffine(degrees=5)` — small rotations simulate camera angle variation
- `ColorJitter` — handles day/night game lighting differences
- **No horizontal flip** — flipping would change the geometry from R-pitcher/R-batter to L-pitcher/L-batter, producing incorrect training signal
- **No vertical flip** — gravity direction is a real signal

In [ ]:
import torch, torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
import json, cv2, numpy as np
from pathlib import Path

class SwingDataset(Dataset):
    """
    Loads 3-frame temporal stacks (9-channel) with launch_angle regression labels.
    Frame order: [t-2 | t | t+2] stacked channel-wise → 9 channels.
    """
    MEAN = [0.485,0.456,0.406] * 3   # repeat ImageNet mean for all 3 frames
    STD  = [0.229,0.224,0.225] * 3

    def __init__(self, root: Path, split: str = 'train'):
        self.root     = root / split
        self.labels   = sorted(self.root.glob('*_label.json'))
        self.is_train = (split == 'train')

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        lp   = self.labels[idx]
        stem = lp.stem.replace('_label', '')
        with open(lp) as f: meta = json.load(f)

        frames = []
        for tag in ['tm2', 't0', 'tp2']:
            img = cv2.imread(str(self.root / f"{stem}_{tag}.jpg"))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            frames.append(img)

        # Stack → (H, W, 9)
        stacked = np.concatenate(frames, axis=2).astype(np.float32) / 255.0

        # Normalize per-channel group
        for i, (m, s) in enumerate(zip(self.MEAN, self.STD)):
            stacked[:, :, i] = (stacked[:, :, i] - m) / s

        # To tensor (9, H, W)
        tensor = torch.from_numpy(stacked.transpose(2, 0, 1))

        if self.is_train:
            # Spatial augmentation (same transform applied to all 3 frames simultaneously
            # since they're already stacked)
            if torch.rand(1) < 0.5:
                # Random crop: take a 224×224 region from the 256×256 patch
                i = torch.randint(0, 32, (1,)).item()
                j = torch.randint(0, 32, (1,)).item()
                tensor = tensor[:, i:i+224, j:j+224]
            else:
                # Centre crop
                tensor = tensor[:, 16:240, 16:240]
            # Small rotation via affine (applied post-stack — consistent across frames)
            if torch.rand(1) < 0.4:
                angle = (torch.rand(1).item() - 0.5) * 10   # ±5 degrees
                tensor = T.functional.rotate(tensor, angle)
        else:
            tensor = tensor[:, 16:240, 16:240]  # centre crop for val

        return tensor, torch.tensor([meta['launch_angle']], dtype=torch.float32)


class SwingAngleCNN(nn.Module):
    """
    ResNet-18 modified for 9-channel temporal input and launch angle regression.

    The first conv layer is expanded to 9 channels by tiling the pretrained
    3-channel weights 3 times and dividing by 3 — this preserves the pretrained
    feature detector scales while accepting temporal stacks.
    """
    def __init__(self, pretrained: bool = True):
        super().__init__()
        backbone = models.resnet18(
            weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        )
        # Expand first conv from 3 → 9 channels
        orig_conv = backbone.conv1
        new_conv  = nn.Conv2d(9, 64, kernel_size=7, stride=2, padding=3, bias=False)
        with torch.no_grad():
            # Tile pretrained weights across the 3 temporal groups and scale
            new_conv.weight[:] = orig_conv.weight.repeat(1, 3, 1, 1) / 3.0
        backbone.conv1 = new_conv

        self.features = nn.Sequential(*list(backbone.children())[:-1])  # → (B,512,1,1)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(256, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.head(self.features(x))


DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

for split in ('train','val'):
    ds = SwingDataset(DATASET_DIR, split)
    print(f'  {split}: {len(ds)} samples')

## 9b. Save session to Drive ← run this now to avoid re-running MediaPipe

Saves everything computed up to this point so a future session can skip
directly to training without re-running YOLO or MediaPipe.

**What gets saved:**

| File | Contents | Size |
|---|---|---|
| `outputs/<stem>_result.json` | SwingResult per video (angles, zones, confidence) | ~2 KB each |
| `outputs/<stem>_landmarks.json` | Full per-frame landmark cache | ~0.5-2 MB each |
| `outputs/batch_results.csv` | All results in one table | ~50 KB |
| `cnn_dataset/train/` | Patch images + labels | already on Drive |
| `cnn_dataset/val/`   | Patch images + labels | already on Drive |

After this cell completes, restart the runtime and run only cells 1–5, then
jump to the **"Restore session"** cell below to reload everything in ~10 seconds.

In [ ]:
import json, csv
from dataclasses import asdict
from tqdm import tqdm

# ── Save landmark cache (the piece not yet on Drive) ─────────────────────────
# _result.json is already written by cell 6 (MediaPipe stage).
# _landmarks.json is the new file — it stores per-frame wrist/body landmarks
# so MediaPipe never needs to re-run on these videos.

def _landmarks_to_list(frame_results):
    """Serialise frame_results → plain list so json.dump can handle it."""
    out = []
    for fi, lms in frame_results:
        if lms is None:
            out.append({'fi': fi, 'lms': None})
        else:
            out.append({
                'fi':  fi,
                'lms': [[lm.x, lm.y, lm.z, lm.visibility] for lm in lms]
            })
    return out

saved_lm, skipped_lm = 0, 0
for result, frame_results, vp, _ in tqdm(swing_results, desc='Saving landmark cache'):
    lm_path = OUTPUT_DIR / (vp.stem + '_landmarks.json')
    if lm_path.exists():
        skipped_lm += 1
        continue
    try:
        with open(lm_path, 'w') as f:
            json.dump(_landmarks_to_list(frame_results), f)
        saved_lm += 1
    except Exception as e:
        print(f'  WARNING: could not save landmarks for {vp.name}: {e}')

print(f'Landmark cache: {saved_lm} saved, {skipped_lm} already existed')

# ── Verify _result.json exists for every video (re-write if missing) ──────────
saved_r, skipped_r = 0, 0
for result, _, vp, _ in swing_results:
    rj = OUTPUT_DIR / (vp.stem + '_result.json')
    if rj.exists():
        skipped_r += 1
        continue
    with open(rj, 'w') as f:
        json.dump(asdict(result), f, indent=2)
    saved_r += 1
print(f'Result JSONs   : {saved_r} written, {skipped_r} already existed')

# ── Summary of what's now on Drive ───────────────────────────────────────────
n_lm = sum(1 for v in all_videos if (OUTPUT_DIR/(v.stem+'_landmarks.json')).exists())
n_rj = sum(1 for v in all_videos if (OUTPUT_DIR/(v.stem+'_result.json')).exists())
n_train = len(list((DATASET_DIR/'train').glob('*_label.json')))
n_val   = len(list((DATASET_DIR/'val').glob('*_label.json')))

print()
print('=== Session saved to Drive ===')
print(f'  Result JSONs      : {n_rj}/{len(all_videos)} videos')
print(f'  Landmark caches   : {n_lm}/{len(all_videos)} videos')
print(f'  Train patches     : {n_train}')
print(f'  Val patches       : {n_val}')
print(f'  Output folder     : {OUTPUT_DIR}')
print()
print('Next session: run cells 1-5, then the Restore Session cell to reload in ~10s.')

## 9c. Restore session ← run this after a restart (instead of cells 6-9)

Reloads `swing_results` and `samples` from Drive in ~10 seconds.
Requires that cell 9b was run at least once.

In [ ]:
import json
from tqdm import tqdm

# Must have already run cells 1-5 (mount, install, write scripts, download model)
import sys
sys.path.insert(0, '/content')
import mediapipe_processing as mp_mod

def _list_to_landmarks(data):
    Lm = mp_mod.Landmark3D
    out = []
    for entry in data:
        fi  = entry['fi']
        raw = entry['lms']
        if raw is None:
            out.append((fi, None))
        else:
            out.append((fi, [Lm(r[0], r[1], r[2], r[3]) for r in raw]))
    return out

swing_results = []
missing       = []

for vp in tqdm(all_videos, desc='Restoring session'):
    rj  = OUTPUT_DIR / (vp.stem + '_result.json')
    lmj = OUTPUT_DIR / (vp.stem + '_landmarks.json')

    if not rj.exists() or not lmj.exists():
        missing.append(vp.name)
        continue

    try:
        with open(rj)  as f: result_dict = json.load(f)
        with open(lmj) as f: lm_data     = json.load(f)

        result = mp_mod.SwingResult(**{
            k: v for k, v in result_dict.items()
            if k in mp_mod.SwingResult.__dataclass_fields__
        })
        frame_results = _list_to_landmarks(lm_data)
        sidecar       = mp_mod.load_sidecar(vp)
        swing_results.append((result, frame_results, vp, sidecar))

    except Exception as e:
        print(f'  Failed to restore {vp.name}: {e}')
        missing.append(vp.name)

# Also rebuild samples list from the saved patch files on Drive
import cv2
samples = []
for split in ('train', 'val'):
    split_dir = DATASET_DIR / split
    for lp in sorted(split_dir.glob('*_label.json')):
        stem = lp.stem.replace('_label', '')
        with open(lp) as f: meta = json.load(f)
        patches = {}
        ok = True
        for tag in ('tm2', 't0', 'tp2'):
            img_path = split_dir / f'{stem}_{tag}.jpg'
            if img_path.exists():
                patches[tag] = cv2.imread(str(img_path))
            else:
                ok = False; break
        if ok:
            samples.append({
                'stem': stem, 'patches': patches,
                'launch_angle': meta['launch_angle'],
                'confidence':   meta.get('confidence', 1.0),
            })

import numpy as np
print(f'\n=== Session restored ===')
print(f'  swing_results : {len(swing_results)} videos loaded')
print(f'  samples       : {len(samples)} patches loaded')
if missing:
    print(f'  Missing ({len(missing)} — need MediaPipe re-run): {missing[:5]}{"..." if len(missing)>5 else ""}')
else:
    print(f'  All videos restored from cache.')

if swing_results:
    angles = [r.launch_angle for r,_,_,_ in swing_results]
    print(f'  Angle range   : {min(angles):.1f} to {max(angles):.1f} deg  (mean {np.mean(angles):.1f})')
print('\nReady — jump to cell 10 (Train) or later.')

## 10. Train `cnn_R.pt`

In [ ]:
import torch, json, numpy as np
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from tqdm import tqdm

EPOCHS     = 50
BATCH_SIZE = 16
LR         = 3e-4
WD         = 1e-4

print('Training single R-handed CNN...')

train_ds = SwingDataset(DATASET_DIR, 'train')
val_ds   = SwingDataset(DATASET_DIR, 'val')
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=(DEVICE=='cuda'))
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=(DEVICE=='cuda'))

model     = SwingAngleCNN(pretrained=True).to(DEVICE)
criterion = nn.SmoothL1Loss(beta=3.0)   # beta=3: less aggressive than default on ±3deg noise
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WD)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

history = {'train_loss':[], 'val_loss':[], 'val_mae':[]}
best_mae = float('inf')

for epoch in range(1, EPOCHS+1):
    # Train
    model.train()
    tl = []
    for imgs, angles in train_dl:
        imgs, angles = imgs.to(DEVICE), angles.to(DEVICE)
        loss = criterion(model(imgs), angles)
        optimizer.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        tl.append(loss.item())
    scheduler.step()

    # Validate
    model.eval()
    vl, preds, targets = [], [], []
    with torch.no_grad():
        for imgs, angles in val_dl:
            imgs, angles = imgs.to(DEVICE), angles.to(DEVICE)
            p = model(imgs)
            vl.append(criterion(p, angles).item())
            preds.extend(p.cpu().numpy().flatten())
            targets.extend(angles.cpu().numpy().flatten())

    tl_mean = np.mean(tl)
    vl_mean = np.mean(vl)
    mae     = float(np.mean(np.abs(np.array(preds)-np.array(targets))))
    history['train_loss'].append(tl_mean)
    history['val_loss'].append(vl_mean)
    history['val_mae'].append(mae)

    if mae < best_mae:
        best_mae = mae
        torch.save(model.state_dict(), CNN_R_PATH)

    if epoch % 5 == 0 or epoch == 1:
        print(f'  Epoch {epoch:3d}/{EPOCHS} | train={tl_mean:.3f} | val={vl_mean:.3f} | MAE={mae:.2f}deg | lr={scheduler.get_last_lr()[0]:.2e}')

print(f'\nBest val MAE: {best_mae:.2f}deg  →  {CNN_R_PATH}')
with open(OUTPUT_DIR/'training_history.json','w') as f: json.dump(history, f, indent=2)

## 11. Training curves

In [ ]:
import matplotlib.pyplot as plt, json
import numpy as np

with open(OUTPUT_DIR/'training_history.json') as f:
    history = json.load(f)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('cnn_R training — 9-channel temporal ResNet-18', fontsize=13)

epochs = range(1, len(history['train_loss'])+1)
axes[0].plot(epochs, history['train_loss'], label='Train')
axes[0].plot(epochs, history['val_loss'],   label='Val')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('SmoothL1 Loss')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs, history['val_mae'], color='darkorange')
axes[1].axhline(y=min(history['val_mae']), color='red', linestyle='--',
                label=f"Best = {min(history['val_mae']):.2f} deg")
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('MAE (degrees)')
axes[1].set_title('Validation MAE'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR/'training_curve.png'), dpi=130, bbox_inches='tight')
plt.show()
print(f"Best MAE: {min(history['val_mae']):.2f} degrees at epoch {np.argmin(history['val_mae'])+1}")

## 12. Evaluation

Loads best weights, runs on the validation set, and reports:
- MAE and RMSE in degrees
- Predicted vs actual scatter plot
- Landing zone classification accuracy (7-zone rule table)
- Per-zone error breakdown

In [ ]:
import torch, json, numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

ANGLE_ZONES = [
    (-90,-10,'ground_ball','Ground ball'),
    (-10,  5,'hard_liner', 'Hard liner'),
    (  5, 15,'line_drive', 'Line drive'),
    ( 15, 25,'solid_fly',  'Solid fly'),
    ( 25, 35,'home_run_window','HR window'),
    ( 35, 50,'high_fly',   'High fly'),
    ( 50, 90,'popup',      'Pop-up'),
]
def angle_to_zone(a):
    for lo,hi,key,_ in ANGLE_ZONES:
        if lo<=a<hi: return key
    return 'popup' if a>=50 else 'ground_ball'

# Load best model
model = SwingAngleCNN(pretrained=False).to(DEVICE)
model.load_state_dict(torch.load(CNN_R_PATH, map_location=DEVICE))
model.eval()

val_ds = SwingDataset(DATASET_DIR, 'val')
val_dl = DataLoader(val_ds, batch_size=16, shuffle=False)

preds, targets = [], []
with torch.no_grad():
    for imgs, angles in val_dl:
        p = model(imgs.to(DEVICE))
        preds.extend(p.cpu().numpy().flatten())
        targets.extend(angles.cpu().numpy().flatten())

preds   = np.array(preds)
targets = np.array(targets)
mae     = float(np.mean(np.abs(preds-targets)))
rmse    = float(np.sqrt(np.mean((preds-targets)**2)))
zone_acc= np.mean([angle_to_zone(p)==angle_to_zone(t) for p,t in zip(preds,targets)])

print(f'Val samples : {len(preds)}')
print(f'MAE         : {mae:.2f} deg')
print(f'RMSE        : {rmse:.2f} deg')
print(f'Zone acc    : {zone_acc:.1%}')

# Per-zone breakdown
print('\nPer-zone MAE:')
for _,_,key,desc in ANGLE_ZONES:
    mask = [angle_to_zone(t)==key for t in targets]
    if sum(mask) > 0:
        zone_mae = float(np.mean(np.abs(preds[mask]-targets[mask])))
        print(f'  {desc:<20s}: {zone_mae:.1f} deg  (n={sum(mask)})')

# Scatter plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f'cnn_R evaluation  |  MAE={mae:.1f}deg  |  Zone={zone_acc:.1%}')

lim = max(abs(targets).max(), abs(preds).max())+5
axes[0].scatter(targets, preds, alpha=0.7, s=40, edgecolors='k', linewidths=0.3)
axes[0].plot([-lim,lim],[-lim,lim],'r--',lw=1,label='Perfect')
axes[0].set_xlabel('True angle (deg)'); axes[0].set_ylabel('Predicted (deg)')
axes[0].set_title('Predicted vs True'); axes[0].legend(); axes[0].grid(alpha=0.3)

errors = preds - targets
axes[1].hist(errors, bins=20, color='steelblue', edgecolor='k', alpha=0.8)
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel('Prediction error (deg)'); axes[1].set_ylabel('Count')
axes[1].set_title(f'Error distribution  |  bias={np.mean(errors):.1f}deg')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR/'eval_R.png'), dpi=130, bbox_inches='tight')
plt.show()

with open(OUTPUT_DIR/'eval_R.json','w') as f:
    json.dump({'mae':mae,'rmse':rmse,'zone_acc':zone_acc,'n':len(preds)}, f, indent=2)

## 12b. Export a sample video where the CNN prediction is correct

Finds the first validation sample where the **predicted landing zone matches the true landing zone** (`zone_match = True`), then renders a side-by-side video showing:
- The original annotated clip (skeleton + contact marker from MediaPipe)
- An overlay panel showing the true angle, predicted angle, and zone result

The output is saved to Drive as `correct_prediction_sample.mp4` and also displayed inline in Colab.

In [ ]:
import cv2, json, math, numpy as np, torch, shutil, subprocess
from pathlib import Path
from IPython.display import Video, display
from torch.utils.data import DataLoader

# ── 1. Find first val sample where zone prediction is correct ─────────────────
val_ds_indexed = SwingDataset(DATASET_DIR, 'val')
val_dl_single  = DataLoader(val_ds_indexed, batch_size=1, shuffle=False)

model.eval()
correct_idx = correct_pred = correct_true = None

for idx, (img_tensor, angle_tensor) in enumerate(val_dl_single):
    with torch.no_grad():
        pred_angle = float(model(img_tensor.to(DEVICE)).cpu().item())
    true_angle = float(angle_tensor.item())
    if angle_to_zone(pred_angle) == angle_to_zone(true_angle):
        correct_idx  = idx
        correct_pred = pred_angle
        correct_true = true_angle
        break

if correct_idx is None:
    print("No correct zone predictions found — model may need more training epochs.")
else:
    label_path = val_ds_indexed.labels[correct_idx]
    stem       = label_path.stem.replace('_label', '')
    print(f"Correct sample  : {stem}")
    print(f"True angle      : {correct_true:+.1f} deg  ({angle_to_zone(correct_true)})")
    print(f"Predicted angle : {correct_pred:+.1f} deg  ({angle_to_zone(correct_pred)})")

    # ── 2. Find source video ──────────────────────────────────────────────────
    source_video = source_result = source_frames = source_sidecar = None
    for result, frame_results, vp, sidecar in swing_results:
        if vp.stem == stem:
            source_video   = vp
            source_result  = result
            source_frames  = frame_results
            source_sidecar = sidecar
            break

    if source_video is None:
        print(f"Source video not found for '{stem}' — re-run MediaPipe or restore session.")
    else:
        out_path  = OUTPUT_DIR / 'correct_prediction_sample.mp4'
        h264_path = OUTPUT_DIR / 'correct_prediction_sample_h264.mp4'

        import mediapipe_processing as mp_mod
        cap, meta = mp_mod.open_video(source_video)
        fps = meta['fps']; w = meta['width']; h = meta['height']
        cf  = source_result.contact_frame

        WINDOW      = 120
        start_frame = max(0, cf - WINDOW)
        end_frame   = min(meta['total_frames'] - 1, cf + WINDOW)
        PANEL_H     = 96

        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        writer = cv2.VideoWriter(str(out_path), fourcc, fps, (w, h + PANEL_H))

        lms_map   = {fi: lms for fi, lms in source_frames}
        bat_boxes = {}
        if source_sidecar and 'bat_boxes' in source_sidecar:
            bat_boxes = {int(k): v for k, v in source_sidecar['bat_boxes'].items()}

        # Contact pixel from landmarks
        c_lms      = lms_map.get(cf)
        contact_px = None
        if c_lms and len(c_lms) >= 66:
            contact_px = mp_mod._contact_pixel(
                c_lms, source_result.batter_handedness, w, h,
                source_result.ball_contact_px
            )

        ANGLE_ZONES = [
            (-90,-10,'Ground ball'),(-10,5,'Hard liner'),(5,15,'Line drive'),
            (15,25,'Solid fly'),(25,35,'HR window'),(35,50,'High fly'),(50,90,'Pop-up'),
        ]
        true_zone = next((l for lo,hi,l in ANGLE_ZONES if lo<=correct_true<hi),
                         'Pop-up' if correct_true>=50 else 'Ground ball')
        pred_zone = next((l for lo,hi,l in ANGLE_ZONES if lo<=correct_pred<hi),
                         'Pop-up' if correct_pred>=50 else 'Ground ball')

        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

        for fi in range(start_frame, end_frame + 1):
            ret, frame = cap.read()
            if not ret: break

            out      = frame.copy()
            age      = fi - cf          # <0 pre-contact, 0 contact, >0 post
            is_post  = age >= 0
            font     = cv2.FONT_HERSHEY_SIMPLEX

            # ── Skeleton from MediaPipe landmarks ─────────────────────────────
            lms = lms_map.get(fi)
            if lms is not None and len(lms) >= 66:
                OFF  = 33
                CONN = [
                    ('l_shoulder','r_shoulder'),('l_shoulder','l_elbow'),
                    ('l_elbow','l_wrist'),('r_shoulder','r_elbow'),
                    ('r_elbow','r_wrist'),('l_shoulder','l_hip'),
                    ('r_shoulder','r_hip'),('l_hip','r_hip'),
                    ('l_hip','l_knee'),('r_hip','r_knee'),
                    ('l_knee','l_ankle'),('r_knee','r_ankle'),
                ]
                LM = mp_mod.LM
                col = (80, 200, 80) if not is_post else (100, 160, 255)
                for ak, bk in CONN:
                    a = lms[OFF+LM[ak]]; b = lms[OFF+LM[bk]]
                    ax2,ay2=int(a.x*w),int(a.y*h); bx2,by2=int(b.x*w),int(b.y*h)
                    if all(0<=v<lim for v,lim in [(ax2,w),(bx2,w),(ay2,h),(by2,h)]):
                        cv2.line(out,(ax2,ay2),(bx2,by2),col,2,cv2.LINE_AA)
                for k in LM:
                    lm=lms[OFF+LM[k]]; px2,py2=int(lm.x*w),int(lm.y*h)
                    if 0<=px2<w and 0<=py2<h:
                        cv2.circle(out,(px2,py2),4,col,-1,cv2.LINE_AA)

            # ── Contact-frame flash ───────────────────────────────────────────
            if age == 0:
                flash = out.copy(); flash[:] = (255, 255, 255)
                cv2.addWeighted(flash, 0.35, out, 0.65, 0, out)
                label_txt = ">>> CONTACT FRAME <<<"
                tw, _ = cv2.getTextSize(label_txt, font, 1.1, 3)[0]
                tx = (w - tw) // 2
                cv2.putText(out, label_txt, (tx+2,52), font, 1.1, (0,0,0),    4, cv2.LINE_AA)
                cv2.putText(out, label_txt, (tx,  50), font, 1.1, (0,220,255), 3, cv2.LINE_AA)

            # ── Contact marker rings (post-contact only) ──────────────────────
            if is_post and contact_px is not None:
                cx_c, cy_c = contact_px
                alpha = max(0.0, 1.0 - age / 40.0)
                if alpha > 0:
                    for rr in [14, 26, 42]:
                        ov = out.copy()
                        cv2.circle(ov,(cx_c,cy_c),rr,(0,80,255),2,cv2.LINE_AA)
                        cv2.addWeighted(ov,alpha,out,1-alpha,0,out)
                cv2.circle(out,(cx_c,cy_c),6,(0,80,255),-1,cv2.LINE_AA)

            # ── Launch angle ray — ONLY at/after contact ──────────────────────
            if is_post and contact_px is not None:
                cx_c, cy_c = contact_px
                # True angle ray (MediaPipe label) — cyan
                ang_true = math.radians(correct_true)
                ex_t = int(cx_c + 200*math.cos(ang_true))
                ey_t = int(cy_c - 200*math.sin(ang_true))
                cv2.arrowedLine(out,(cx_c,cy_c),(ex_t,ey_t),(0,220,220),2,cv2.LINE_AA,tipLength=0.10)
                cv2.putText(out,f"TRUE {correct_true:+.1f}d",(ex_t+4,ey_t-6),
                            font,0.55,(0,220,220),1,cv2.LINE_AA)
                # CNN predicted angle ray — yellow
                ang_pred = math.radians(correct_pred)
                ex_p = int(cx_c + 200*math.cos(ang_pred))
                ey_p = int(cy_c - 200*math.sin(ang_pred))
                cv2.arrowedLine(out,(cx_c,cy_c),(ex_p,ey_p),(0,220,255),2,cv2.LINE_AA,tipLength=0.10)
                cv2.putText(out,f"PRED {correct_pred:+.1f}d",(ex_p+4,ey_p+14),
                            font,0.55,(0,220,255),1,cv2.LINE_AA)

            # ── Bottom panel — ONLY shown at/after contact ────────────────────
            panel = np.zeros((PANEL_H, w, 3), dtype=np.uint8)
            panel[:] = (18, 18, 28)

            if is_post:
                # Green border = correct prediction
                panel[0:3,:]  = (0,200,80); panel[-3:,:] = (0,200,80)
                panel[:,0:3]  = (0,200,80); panel[:,-3:] = (0,200,80)
                cv2.putText(panel,
                    f"TRUE:  {correct_true:+.1f} deg  [{true_zone}]",
                    (18,28), font, 0.60, (220,220,220), 1, cv2.LINE_AA)
                cv2.putText(panel,
                    f"PRED:  {correct_pred:+.1f} deg  [{pred_zone}]",
                    (18,56), font, 0.60, (0,220,255), 1, cv2.LINE_AA)
                cv2.putText(panel, "ZONE MATCH: TRUE",
                    (18,82), font, 0.55, (60,220,80), 1, cv2.LINE_AA)
                cv2.putText(panel,
                    f"Video: {stem[:50]}",
                    (w//2+10, 28), font, 0.46, (160,160,160), 1, cv2.LINE_AA)
                cv2.putText(panel,
                    f"Conf: {source_result.confidence:.0%}   Hand: {source_result.batter_handedness}",
                    (w//2+10, 56), font, 0.46, (160,160,160), 1, cv2.LINE_AA)
                cv2.putText(panel,
                    f"frame {fi}  (contact={cf})",
                    (w//2+10, 82), font, 0.40, (110,110,110), 1, cv2.LINE_AA)
            else:
                # Pre-contact — grey panel, countdown
                secs_to = (cf - fi) / fps
                cv2.putText(panel,
                    f"Pre-contact  —  {secs_to:.2f}s to contact frame {cf}",
                    (18, 52), font, 0.52, (130,130,130), 1, cv2.LINE_AA)
                cv2.putText(panel,
                    "Launch angle will appear at contact",
                    (18, 78), font, 0.44, (90,90,90), 1, cv2.LINE_AA)

            combined = np.vstack([out, panel])
            writer.write(combined)

        cap.release()
        writer.release()
        print(f"Video saved: {out_path}")

        if shutil.which('ffmpeg'):
            subprocess.run([
                'ffmpeg','-y','-i',str(out_path),
                '-vcodec','libx264','-crf','23','-preset','fast',
                '-acodec','aac', str(h264_path)
            ], capture_output=True)
            display(Video(str(h264_path), embed=True, width=960))
        else:
            display(Video(str(out_path), embed=True, width=960))

## 13. Inference — predict on a new video

Loads `cnn_R.pt` and predicts launch angle from the 3-frame temporal stack
centred on the detected contact frame. No pose estimation required at inference time
— just the contact frame detection + patch extraction.

In [ ]:
import torch, cv2, json, numpy as np
import torchvision.transforms.functional as TF
from pathlib import Path

def _contact_frame_from_video(video_path: Path, fps: float, total: int) -> int:
    """Fast motion-energy contact frame detection (no pose required)."""
    from scipy.signal import savgol_filter
    cap   = cv2.VideoCapture(str(video_path))
    energies, prev_gray = [], None
    for fi in range(0, total, 2):
        cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
        ret, frame = cap.read()
        if not ret: break
        gray = cv2.GaussianBlur(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), (5,5), 0)
        if prev_gray is not None:
            energies.append((fi, float(np.mean(cv2.absdiff(gray, prev_gray)))))
        prev_gray = gray
    cap.release()
    if len(energies) < 10:
        return total // 2
    idxs  = np.array([e[0] for e in energies])
    vals  = np.array([e[1] for e in energies])
    sm    = savgol_filter(vals, min(15, len(vals)-(1-len(vals)%2)), 3)
    n     = len(sm)
    gated = sm.copy()
    gated[:int(n*0.30)]=0; gated[int(n*0.85):]=0
    pi    = int(np.argmax(gated))
    ci    = pi
    thr   = sm[pi]*0.70
    for i in range(1, len(sm)-pi-1):
        p=pi+i
        if sm[p]<sm[p-1] and sm[p]<sm[p+1] and sm[p]<thr:
            ci=p; break
    return int(idxs[ci])


def predict_launch_angle(video_path: str) -> dict:
    """
    Predict launch angle for a single R-angle swing video.
    Returns dict with launch_angle_cnn, landing_zone, contact_frame, contact_time_s.
    """
    vp  = Path(video_path)
    cap = cv2.VideoCapture(str(vp))
    fps   = cap.get(cv2.CAP_PROP_FPS) or 60.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w_px  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h_px  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    # Try loading pre-computed contact frame from result JSON
    cf = None
    rj = OUTPUT_DIR / (vp.stem + '_result.json')
    if rj.exists():
        with open(rj) as f: cf = json.load(f).get('contact_frame')
    if cf is None:
        cf = _contact_frame_from_video(vp, fps, total)

    # Extract 3-frame stack centred at contact
    cx, cy = w_px // 2, h_px // 2   # centre heuristic; replace with pose if available
    frames = []
    cap = cv2.VideoCapture(str(vp))
    for offset in [-2, 0, 2]:
        fi = max(0, min(total-1, cf + offset))
        cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
        ret, frame = cap.read()
        if not ret: frame = np.zeros((h_px, w_px, 3), dtype=np.uint8)
        frames.append(extract_patch(frame, cx, cy, PATCH_SIZE))
    cap.release()

    stacked = np.concatenate([cv2.cvtColor(f, cv2.COLOR_BGR2RGB) for f in frames], axis=2)
    stacked = stacked.astype(np.float32) / 255.0
    MEAN = [0.485,0.456,0.406]*3; STD=[0.229,0.224,0.225]*3
    for i,(m,s) in enumerate(zip(MEAN,STD)):
        stacked[:,:,i]=(stacked[:,:,i]-m)/s
    tensor = torch.from_numpy(stacked.transpose(2,0,1)).unsqueeze(0)
    # Centre crop to 224×224
    tensor = tensor[:, :, 16:240, 16:240]

    model_inf = SwingAngleCNN(pretrained=False).to(DEVICE)
    model_inf.load_state_dict(torch.load(CNN_R_PATH, map_location=DEVICE))
    model_inf.eval()
    with torch.no_grad():
        angle = float(model_inf(tensor.to(DEVICE)).cpu().item())

    return {
        'launch_angle_cnn': round(angle, 2),
        'landing_zone':     angle_to_zone(angle),
        'contact_frame':    cf,
        'contact_time_s':   round(cf / fps, 3),
    }


# ── Run on all processed videos and compare to MediaPipe ─────────────────────
import pandas as pd

rows = []
for result, _, vp, _ in swing_results:
    pred = predict_launch_angle(str(vp))
    rows.append({
        'video':          vp.stem,
        'mediapipe_angle':result.launch_angle,
        'cnn_angle':      pred['launch_angle_cnn'],
        'error_deg':      abs(result.launch_angle - pred['launch_angle_cnn']),
        'mp_zone':        angle_to_zone(result.launch_angle),
        'cnn_zone':       pred['landing_zone'],
        'zone_match':     angle_to_zone(result.launch_angle) == pred['landing_zone'],
        'contact_t':      pred['contact_time_s'],
    })

df = pd.DataFrame(rows)
print(df[['video','mediapipe_angle','cnn_angle','error_deg','zone_match']].to_string())
print(f"\nMean |error|: {df['error_deg'].mean():.2f} deg")
print(f"Zone match  : {df['zone_match'].mean():.1%}")
df.to_csv(str(OUTPUT_DIR/'cnn_vs_mediapipe.csv'), index=False)

## 14. Verify Drive outputs

In [ ]:
from pathlib import Path

print('=== Drive output summary ===')
items = {
    'Annotated videos':       list(DRIVE_VIDEO_DIR.rglob('*_annotated.mp4')),
    'JSON results':           list(OUTPUT_DIR.glob('*_result.json')),
    'YOLO sidecars':          [v.with_suffix('.yolo.json') for v in all_videos
                               if v.with_suffix('.yolo.json').exists()],
    'CNN model (cnn_R.pt)':   [CNN_R_PATH] if CNN_R_PATH.exists() else [],
    'Train patches':          list((DATASET_DIR/'train').glob('*_t0.jpg')),
    'Val patches':            list((DATASET_DIR/'val').glob('*_t0.jpg')),
    'Batch CSV':              [OUTPUT_DIR/'batch_results.csv']
                              if (OUTPUT_DIR/'batch_results.csv').exists() else [],
    'CNN vs MediaPipe CSV':   [OUTPUT_DIR/'cnn_vs_mediapipe.csv']
                              if (OUTPUT_DIR/'cnn_vs_mediapipe.csv').exists() else [],
}

for section, files in items.items():
    print(f"  {section:<30s}: {len(files)}")

print(f"\nAll outputs in: {OUTPUT_DIR}")
print('Done!')